# <center>Systematic Trading Strategies with Machine Learning</center>
**<center>Coursework Project – Meta-Labeling for Trading Signal Filtering</center>**
<center>Group 13: Agustina Albez, Garance Danvin, Helene Rabain, Kevin Aoun and Nathan Sebbag</center>

---

## Table of Contents

1. [Project Overview](#1.-Project-Overview)
2. [Project Pipeline](#2.-Project-Pipeline)
3. [Methodological Principles](#3.-Methodological-Principles)
4. [Asset Universe](#4.-Asset-Universe)
5. [Initial Setup](#5.-Initial-Setup)

**[Phase 1 — Data Preparation and Initial Exploration](#Phase-1-—-Data-Preparation-and-Initial-Exploration)**
- [1.1 Initial Data Standardization](#1.1-Initial-Data-Standardization)
- [1.2 Data Integrity Checks](#1.2-Data-Integrity-Checks)
- [1.3 Signal Distribution Analysis](#1.3-Signal-Distribution-Analysis)
- [1.4 OHLC Consistency Checks](#1.4-OHLC-Consistency-Checks)
- [1.5 Price Series Visualization](#1.5-Price-Series-Visualization)
- [1.6 Return Series Construction](#1.6-Return-Series-Construction)
- [1.7 Reshaping Primary Signals](#1.7-Reshaping-Primary-Signals)

**[Phase 2 — Feature Engineering](#Phase-2-—-Feature-Engineering)**
- [2.1 Core Return and Momentum Features](#2.1-Core-Return-and-Momentum-Features)
- [2.2 Volatility Features](#2.2-Volatility-Features)
- [2.2b Higher Moments and Range Position](#2.2b-Higher-Moments-and-Range-Position)
- [2.3 Volume and Open Interest Features](#2.3-Volume-and-Open-Interest-Features)
- [2.3b Microstructure and Liquidity Features](#2.3b-Microstructure-and-Liquidity-Features)
- [2.4 Technical Indicators](#2.4-Technical-Indicators)
- [2.5 Time-Series Dependence Features](#2.5-Time-Series-Dependence-Features)
- [2.5b Spectral and Fractal Features](#2.5b-Spectral-and-Fractal-Features)
- [2.6 Cross-Sectional Features](#2.6-Cross-Sectional-Features)
- [2.6b Seasonality Features](#2.6b-Seasonality-Features)
- [2.7 Inter-energy Spreads](#2.7-Inter-energy-spreads)
- [2.8 Cross-Asset Regime](#2.8-Cross-Asset-Regime)
- [2.9 Metals Ratios](#2.9-Metals-Ratios)
- [2.10 Cross-Asset Correlations](#2.10-Cross-Asset-Correlations)
- [2.11 Cross-Asset Momentum](#2.11-Cross-Asset-Momentum)
- [2.12 Cross-Asset Volatility](#2.12-Cross-Asset-Volatility)
- [2.13 Copper Leading Indicators](#2.13-Copper-Leading-Indicators)
- [2.14 Macro Context](#2.14-Macro-Context)
- [2.15 Feature Set Summary](#2.15-Feature-Set-Summary)
- [2.16 HMM Regime Features](#2.16-HMM-Regime-Features)
- [2.17 Merge Features with Primary Signals](#2.17-Merge-Features-with-Primary-Signals)
- [2.18 Primary Signal Interaction Features](#2.18-Primary-Signal-Interaction-Features)

**[Phase 3 — Triple-Barrier Labeling](#Phase-3-—-Triple-Barrier-Meta-Labeling)**
- [3.1 Methodological Decisions and Design Notes](#Phase-3-—-Triple-Barrier-Labeling:-Methodological-Decisions-and-Design-Notes)
- [3.2 Temporal Structure Check](#3.2-Temporal-Structure-Check)
- [3.3 Choice of Volatility Estimator](#3.3-Choice-of-Volatility-Estimator)
- [3.4 Initial Triple-Barrier Parameters](#3.4-Initial-Triple-Barrier-Parameters)
- [3.5 Triple-Barrier Labeling Function](#3.5-Triple-Barrier-Labeling-Function)
- [3.6 Apply Triple-Barrier Labeling](#3.6-Apply-Triple-Barrier-Labeling)
- [3.7 Remove Unlabelable Observations](#3.7-Remove-Unlabelable-Observations)

**[Phase 4 — Model Development](#Phase-4-—-Model-Development)**
- [4.1 Data Cleaning](#Data-cleaning-meta_labeled_df)
- [4.2 Split into X, y, t1](#Split-into-X,-y,-t1-for-the-metamodel)
- [4.3 Purged K-Fold with Embargo](#Purged-K-Fold-with-embargo)
- [4.4 Logistic Regression](#Logistic-Regression)
- [4.5 Tree-based Models (Random Forest, XGBoost)](#Tree-based-model)
- [4.6 Neural Networks](#Neural-Networks)
- [4.7 Comparison with a Bernoulli Baseline](#Comparison-with-a-Bernoulli)
- [4.8 Side-by-side Comparison](#Side-by-side-comparison)

**[Phase 5 — Feature Importance Analysis (Cluster-Level)](#Phase-5---Feature-Importance-Analysis:-Cluster-Level)**

---

## 1. Project Overview

The objective of this project is to build a **metamodel** on top of a provided primary trading signal across futures contracts from multiple asset classes.

The primary model generates daily trading signals in $\{-1,0,+1\}$:
- $+1$: long signal
- $-1$: short signal
- $0$: no position

Our goal is **not** to predict returns directly.  
Instead, we aim to estimate the probability that a given primary signal is worth taking under a triple-barrier labeling framework.

More formally, we seek to model:

$$
P(\text{Trade is profitable} \mid \text{Features})
$$

using machine learning techniques.

---

## 2. Project Pipeline

The project is structured into the following stages:

### Phase 1 — Data Preparation and Exploration
- Load and clean OHLCV data
- Load and align primary signals
- Verify data integrity and temporal consistency
- Explore the characteristics of each asset class

### Phase 2 — Feature Engineering
Construction of predictive features from market data, including:
- Technical indicators
- Volatility and momentum features
- Cross-sectional features
- Latent regime features (HMM)
- Additional engineered features

### Phase 3 — Triple-Barrier Labeling
Implementation of the triple-barrier method to define supervised learning targets:
- Profit-taking barrier
- Stop-loss barrier
- Maximum holding period

We will justify all parameter choices economically and statistically.

### Phase 4 — Model Development
We will compare several model families:
- Linear models
- Tree-based ensemble methods
- Neural networks
- Ensembling

Hyperparameter tuning and robust validation procedures will be applied.

### Phase 5 — Feature Importance Analysis
We will study:
- Individual feature importance
- Cluster-level feature importance
- SHAP/MDI/MDA

### Phase 6 — Out-of-Sample Evaluation
Evaluation on a clean out-of-sample period using:
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrices
- Comparison against the raw primary signal

### Optional Phase 7 — Strategy Construction
(Optional competition track)

Construction of a position-sizing strategy using metamodel probabilities.

---

## 3. Methodological Principles

Throughout the project, particular attention will be paid to:
- Avoiding look-ahead bias
- Preventing data leakage
- Using time-aware validation procedures
- Ensuring reproducibility
- Maintaining economic interpretability of results

---

## 4. Asset Universe

The project covers futures contracts from three asset classes:

### Equity Index Futures
- ES1S — S&P 500
- NQ1S — Nasdaq 100
- FESX1S — Euro Stoxx 50

### Energy
- CL1S — WTI Crude Oil
- HO1S — Heating Oil
- RB1S — RBOB Gasoline
- NG1S — Natural Gas

### Metals
- GC1S — Gold
- SI1S — Silver
- HG1S — Copper
- PL1S — Platinum

For this project, we focus on the energy assets.


# Phase 1 — Data Preparation and Initial Exploration

In this first phase, we prepare the raw datasets for the meta-labeling pipeline.

The objectives of this phase are:

1. Load the OHLCV dataset and the primary signals dataset.
2. Standardize dates, instrument names, and column formats.
3. Filter the universe to Energy instruments only.
4. Align market data and primary signals on a common trading calendar.
5. Perform basic data integrity checks:
   - missing values,
   - duplicated rows,
   - date coverage,
   - signal distribution,
   - price and volume sanity checks.
6. Prepare a clean base dataframe that will later be used for feature engineering and triple-barrier labeling.

At this stage, we do **not** create labels yet.  
The purpose is to build a clean, reliable data foundation before moving to feature engineering.

In [1]:
# ============================================================
# Phase 1 — Setup and Data Loading
# ============================================================

import os
import numpy as np
from numpy import lib
import pandas as pd

import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from sklearn.decomposition import PCA
from hmmlearn.hmm import GaussianHMM

import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.dates as mdates

import statsmodels.api as sm1
import talib
from hmmlearn.hmm import GaussianHMM
from sklearn.mixture import GaussianMixture

import shap

import itertools

import yfinance as yf

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from itertools import product
import statsmodels.tsa.stattools as ts

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from itertools import product

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin
import copy

from sklearn.metrics import (
    roc_auc_score, log_loss, accuracy_score,
    precision_score, recall_score, f1_score,
)

# Display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [2]:
# ------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------

DATA_DIR = "."  # Change this path if your CSV files are in another folder

OHLCV_FILE = os.path.join(DATA_DIR, "ohlcv_data.csv")
SIGNALS_FILE = os.path.join(DATA_DIR, "primary_signals.csv")

ENERGY_INSTRUMENTS = ["cl1s", "ho1s", "rb1s", "ng1s"]

INSTRUMENT_NAMES = {
    "cl1s": "WTI Crude Oil",
    "ho1s": "Heating Oil",
    "rb1s": "RBOB Gasoline",
    "ng1s": "Natural Gas",
}

print("Energy universe:")
for ticker, name in INSTRUMENT_NAMES.items():
    print(f"- {ticker.upper()}: {name}")

Energy universe:
- CL1S: WTI Crude Oil
- HO1S: Heating Oil
- RB1S: RBOB Gasoline
- NG1S: Natural Gas


In [3]:
# ============================================================
# Load Raw Data
# ============================================================

ohlcv_raw = pd.read_csv(OHLCV_FILE)
signals_raw = pd.read_csv(SIGNALS_FILE)

print("OHLCV shape:", ohlcv_raw.shape)
print("Signals shape:", signals_raw.shape)

display(ohlcv_raw.head())
display(signals_raw.head())

OHLCV shape: (83547, 8)
Signals shape: (645, 12)


,date,instrument,open,high,low,close,volume,open_interest
0,1990-01-02,cl1s,21.8000,22.920,21.7900,22.8900,22868.0,66308.0
1,1990-01-02,gc1s,401.0000,404.600,400.2000,402.1000,20747.0,68855.0
2,1990-01-02,hg1s,1.0470,1.064,1.0430,1.0620,3325.0,19735.0
3,1990-01-02,ho1s,0.7475,0.776,0.7415,0.7739,20280.0,33732.0
4,1990-01-02,pl1s,484.0000,485.000,479.5000,482.2000,4561.0,13256.0


,date,es1s,nq1s,fesx1s,cl1s,ho1s,rb1s,ng1s,gc1s,si1s,hg1s,pl1s
0,2020-01-03,1,1,-1,0,0,1,0,0,1,1,-1
1,2020-01-06,1,-1,1,0,0,1,0,0,1,1,1
2,2020-01-07,1,-1,1,-1,0,-1,0,0,1,1,1
3,2020-01-08,1,1,1,0,0,1,0,0,1,1,1
4,2020-01-09,1,-1,-1,0,0,1,0,0,1,1,1


## 1.1 Initial Data Standardization

We now standardize the two datasets:

- convert dates to `datetime`,
- convert instrument names to lowercase,
- sort observations chronologically,
- filter the OHLCV data and primary signals to the Energy instruments only.

This ensures that both datasets use the same naming convention and can later be merged safely.

In [4]:
# ============================================================
# Standardize Dates and Instrument Names
# ============================================================

ohlcv = ohlcv_raw.copy()
signals = signals_raw.copy()

# Convert date columns
ohlcv["date"] = pd.to_datetime(ohlcv["date"])
signals["date"] = pd.to_datetime(signals["date"])

# Standardize instrument names
ohlcv["instrument"] = ohlcv["instrument"].str.lower()

# Keep only Energy instruments
ohlcv_energy = ohlcv[ohlcv["instrument"].isin(ENERGY_INSTRUMENTS)].copy()

# Keep date + Energy signal columns
signals_energy = signals[["date"] + ENERGY_INSTRUMENTS].copy()

# Sort
ohlcv_energy = ohlcv_energy.sort_values(["instrument", "date"]).reset_index(drop=True)
signals_energy = signals_energy.sort_values("date").reset_index(drop=True)

print("Energy OHLCV shape:", ohlcv_energy.shape)
print("Energy signals shape:", signals_energy.shape)

display(ohlcv_energy.head())
display(signals_energy.head())

Energy OHLCV shape: (32614, 8)
Energy signals shape: (645, 5)


,date,instrument,open,high,low,close,volume,open_interest
0,1990-01-02,cl1s,21.80,22.92,21.79,22.89,22868.0,66308.0
1,1990-01-03,cl1s,23.20,23.80,23.00,23.68,45177.0,61428.0
2,1990-01-04,cl1s,23.88,23.92,22.83,23.41,50061.0,60995.0
3,1990-01-05,cl1s,23.42,23.70,23.03,23.08,53070.0,57258.0
4,1990-01-08,cl1s,22.60,22.60,21.55,21.62,39720.0,54644.0


,date,cl1s,ho1s,rb1s,ng1s
0,2020-01-03,0,0,1,0
1,2020-01-06,0,0,1,0
2,2020-01-07,-1,0,-1,0
3,2020-01-08,0,0,1,0
4,2020-01-09,0,0,1,0


## 1.2 Data Integrity Checks

Before creating features or labels, we first verify the quality of the raw Energy datasets.

We check:
- date coverage by instrument,
- duplicated observations,
- missing values,
- OHLCV consistency,
- and primary signal distributions.

These checks are important because any data issue at this stage could later create misleading labels, feature leakage, or incorrect model evaluation.

In [5]:
# ============================================================
# Basic Data Integrity Checks
# ============================================================

print("OHLCV date range by instrument:")
display(
    ohlcv_energy
    .groupby("instrument")["date"]
    .agg(["min", "max", "count"])
)

print("\nSignals date range:")
display(
    signals_energy["date"].agg(["min", "max", "count"])
)

print("\nDuplicate OHLCV rows by (date, instrument):")
n_dup_ohlcv = ohlcv_energy.duplicated(subset=["date", "instrument"]).sum()
print(n_dup_ohlcv)

print("\nDuplicate signal dates:")
n_dup_signals = signals_energy.duplicated(subset=["date"]).sum()
print(n_dup_signals)

print("\nMissing values in OHLCV:")
display(ohlcv_energy.isna().sum())

print("\nMissing values in signals:")
display(signals_energy.isna().sum())

OHLCV date range by instrument:


,min,max,count
instrument,,,
cl1s,1990-01-02,2022-06-30,8171
ho1s,1990-01-02,2022-06-30,8169
ng1s,1990-04-04,2022-06-30,8104
rb1s,1990-01-02,2022-06-30,8170



Signals date range:


min      2020-01-03 00:00:00
max      2022-06-30 00:00:00
count                    645
Name: date, dtype: object


Duplicate OHLCV rows by (date, instrument):
0

Duplicate signal dates:
0

Missing values in OHLCV:


date             0
instrument       0
open             0
high             0
low              0
close            0
volume           0
open_interest    0
dtype: int64


Missing values in signals:


date    0
cl1s    0
ho1s    0
rb1s    0
ng1s    0
dtype: int64

## 1.3 Signal Distribution Analysis

Before building labels or training models, we inspect the distribution of the primary signals.

This is important because:
- the dataset may be imbalanced,
- some instruments may trade much more frequently than others,
- and the proportion of long/short/flat signals may affect both labeling and model performance.

We therefore compute the distribution of:
- long signals (+1),
- short signals (-1),
- and inactive periods (0),
for each Energy instrument.

In [6]:
# ============================================================
# Signal Distribution
# ============================================================

signal_distribution = {}

for inst in ENERGY_INSTRUMENTS:
    
    counts = (
        signals_energy[inst]
        .value_counts()
        .sort_index()
    )
    
    signal_distribution[inst] = counts

    print(f"\n{inst.upper()} signal distribution:")
    print(counts)

    print("\nPercentages:")
    print((counts / counts.sum() * 100).round(2))


CL1S signal distribution:
cl1s
-1     36
 0    223
 1    386
Name: count, dtype: int64

Percentages:
cl1s
-1     5.58
 0    34.57
 1    59.84
Name: count, dtype: float64

HO1S signal distribution:
ho1s
-1     10
 0    582
 1     53
Name: count, dtype: int64

Percentages:
ho1s
-1     1.55
 0    90.23
 1     8.22
Name: count, dtype: float64

RB1S signal distribution:
rb1s
-1    261
 0     17
 1    367
Name: count, dtype: int64

Percentages:
rb1s
-1    40.47
 0     2.64
 1    56.90
Name: count, dtype: float64

NG1S signal distribution:
ng1s
-1    124
 0    521
Name: count, dtype: int64

Percentages:
ng1s
-1    19.22
 0    80.78
Name: count, dtype: float64


## 1.4 OHLC Consistency Checks

We now perform several sanity checks on the OHLC data.

For each observation, financial market conventions imply:


$\text{low} \leq \text{open, close} \leq \text{high}$


Violations of these inequalities may indicate:
- corrupted observations,
- bad data adjustments,
- or preprocessing issues.

We also verify that prices and trading activity remain strictly positive.

In [7]:
# ============================================================
# OHLC Consistency Checks
# ============================================================

# High should be >= low
invalid_high_low = (ohlcv_energy["high"] < ohlcv_energy["low"]).sum()

# Open should lie inside [low, high]
invalid_open = (
    (ohlcv_energy["open"] < ohlcv_energy["low"]) |
    (ohlcv_energy["open"] > ohlcv_energy["high"])
).sum()

# Close should lie inside [low, high]
invalid_close = (
    (ohlcv_energy["close"] < ohlcv_energy["low"]) |
    (ohlcv_energy["close"] > ohlcv_energy["high"])
).sum()

# Negative or zero prices
non_positive_prices = (
    (ohlcv_energy[["open", "high", "low", "close"]] <= 0)
    .sum()
    .sum()
)

print("Invalid high/low rows:", invalid_high_low)
print("Invalid open rows:", invalid_open)
print("Invalid close rows:", invalid_close)
print("Non-positive prices:", non_positive_prices)

Invalid high/low rows: 0
Invalid open rows: 0
Invalid close rows: 0
Non-positive prices: 0


## 1.5 Price Series Visualization

We now visualize the historical closing prices of the Energy futures contracts.

The objective is not yet predictive modeling, but rather:
- understanding the long-term behavior of each market,
- identifying volatility regimes,
- detecting structural breaks and crises,
- and building intuition about the data before feature engineering.

Since commodity futures can exhibit very different scales and volatility levels, each instrument is plotted separately.

In [8]:
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[
        f"{inst.upper()} — {INSTRUMENT_NAMES[inst]}"
        for inst in ENERGY_INSTRUMENTS
    ]
)

for row, inst in enumerate(ENERGY_INSTRUMENTS, start=1):
    df_inst = ohlcv_energy[ohlcv_energy["instrument"] == inst]
    
    fig.add_trace(
        go.Scatter(
            x=df_inst["date"],
            y=df_inst["close"],
            mode="lines",
            name=inst.upper(),
            line=dict(width=1.5)
        ),
        row=row,
        col=1
    )
    
    fig.update_yaxes(title_text="Close Price", row=row, col=1)

fig.update_layout(
    height=900,
    width=950,
    title_text="Historical Close Prices for Energy Futures",
    showlegend=False
)

fig.show()

## 1.6 Return Series Construction

Financial machine learning models are generally built on returns rather than raw price levels.

We therefore compute daily log returns for each instrument:

$$
r_t = \log\left(\frac{P_t}{P_{t-1}}\right)
$$

Log returns are preferred because:
- they are additive through time,
- more statistically stable than prices,
- and largely invariant to the absolute price scale.

This is particularly important for futures contracts, where continuous-contract adjustments may distort long-term price levels.

In [9]:
# ============================================================
# Daily Log Returns
# ============================================================

ohlcv_energy["log_return"] = (
    ohlcv_energy
    .groupby("instrument")["close"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

display(
    ohlcv_energy[
        ["date", "instrument", "close", "log_return"]
    ].head(10)
)

,date,instrument,close,log_return
0,1990-01-02,cl1s,22.89,NaN
1,1990-01-03,cl1s,23.68,0.033931
2,1990-01-04,cl1s,23.41,-0.011468
3,1990-01-05,cl1s,23.08,-0.014197
4,1990-01-08,cl1s,21.62,-0.065348
5,1990-01-09,cl1s,22.07,0.020600
6,1990-01-10,cl1s,22.90,0.036918
7,1990-01-11,cl1s,23.14,0.010426
8,1990-01-12,cl1s,23.13,-0.000432
9,1990-01-15,cl1s,22.36,-0.033857


In [10]:
# ============================================================
# Plot Log Returns
# ============================================================
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=[
        f"{inst.upper()} — {INSTRUMENT_NAMES[inst]}"
        for inst in ENERGY_INSTRUMENTS
    ]
)

for row, inst in enumerate(ENERGY_INSTRUMENTS, start=1):
    df_inst = ohlcv_energy[ohlcv_energy["instrument"] == inst]

    fig.add_trace(
        go.Scatter(
            x=df_inst["date"],
            y=df_inst["log_return"],
            mode="lines",
            name=inst.upper(),
            line=dict(width=1.5)
        ),
        row=row,
        col=1
    )

    fig.add_hline(
        y=0,
        line=dict(color="black", width=1),
        row=row,
        col=1
    )

    fig.update_yaxes(title_text="Log Return", row=row, col=1)

fig.update_layout(
    height=900,
    width=950,
    title_text="Daily Log Returns for Energy Futures",
    showlegend=False
)

fig.show()

## 1.7 Reshaping Primary Signals

The primary signal dataset is currently stored in a wide format:

| date | cl1s | ho1s | rb1s | ng1s |
|---|---|---|---|---|

However, the OHLCV dataset is stored in long format:

| date | instrument | open | high | low | close | ... |

To simplify merging and downstream processing, we reshape the signal dataset into long format using `pandas.melt()`.

The resulting structure becomes:

| date | instrument | signal |
|---|---|---|

This representation is much more convenient for:
- feature engineering,
- labeling,
- merging datasets,
- and machine learning pipelines.

In [11]:
# ============================================================
# Reshape Signals to Long Format
# ============================================================

signals_long = signals_energy.melt(
    id_vars="date",
    value_vars=ENERGY_INSTRUMENTS,
    var_name="instrument",
    value_name="primary_signal"
)

signals_long = (
    signals_long
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Signals long shape:", signals_long.shape)

display(signals_long.head(10))

Signals long shape: (2580, 3)


,date,instrument,primary_signal
0,2020-01-03,cl1s,0
1,2020-01-06,cl1s,0
2,2020-01-07,cl1s,-1
3,2020-01-08,cl1s,0
4,2020-01-09,cl1s,0
5,2020-01-10,cl1s,0
6,2020-01-13,cl1s,0
7,2020-01-14,cl1s,0
8,2020-01-15,cl1s,0
9,2020-01-16,cl1s,0


# Phase 2 — Feature Engineering

In this phase, we construct predictive features from the Energy OHLCV dataset.

The objective is to describe the market environment in which each primary signal occurs.  
These features will later be used by the metamodel to estimate whether a given primary trading signal is worth taking.

We construct features separately for each instrument, using only past and current information available at each date.  
This is essential to avoid look-ahead bias.

The feature engineering process is organized into several blocks:

1. **Core return and momentum features**
   - daily returns,
   - rolling cumulative returns,
   - short-term and medium-term momentum.

2. **Volatility features**
   - rolling realized volatility,
   - volatility ratios,
   - volatility regime indicators.

3. **Volume and open-interest features**
   - volume changes,
   - volume z-scores,
   - open-interest changes.

4. **Technical indicators**
   - RSI,
   - MACD,
   - Bollinger-style z-scores.

5. **Time-series dependence features**
   - rolling autocorrelation,
   - volatility clustering proxies,
   - trend persistence measures.

6. **Cross-sectional features**
   - relative momentum,
   - relative volatility,
   - ranks within the Energy asset class.

7. **Latent regime features**
   - HMM-based regime probabilities,
   - market turbulence indicators.

At this stage, we start with a robust set of core statistical features before adding more advanced indicators.

## 2.1 Core Return and Momentum Features

We begin with a set of core return-based and momentum features.

Momentum features aim to capture whether an instrument has recently experienced persistent positive or negative price movements.  
Such features are widely used in systematic trading strategies, where trend-following and persistence effects often play an important role.

For each instrument, we compute rolling momentum over several horizons:

$$\text{mom}_{k,t} = \log\left(\frac{P_t}{P_{t-k}}\right)$$

where:
- $P_t$ is the closing price at time $t$,
- and $k$ represents the lookback horizon.

We use multiple horizons in order to capture:
- short-term momentum,
- medium-term trends,
- and longer-term price persistence.

In addition, we compute rolling average returns:

$$\bar r_t^{(k)} = \frac{1}{k}\sum_{i=0}^{k-1} r_{t-i}$$

which provide smoother estimates of recent market direction and local return persistence.

All features are computed independently for each instrument using:

```python
groupby("instrument")
```

In [12]:
# ============================================================
# Phase 2 — Feature Engineering Setup
# ============================================================

features_df = ohlcv_energy.copy()

# Make sure data is sorted before computing rolling/grouped features
features_df = (
    features_df
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Feature engineering base shape:", features_df.shape)
display(features_df.head())

Feature engineering base shape: (32614, 9)


,date,instrument,open,high,low,close,volume,open_interest,log_return
0,1990-01-02,cl1s,21.80,22.92,21.79,22.89,22868.0,66308.0,NaN
1,1990-01-03,cl1s,23.20,23.80,23.00,23.68,45177.0,61428.0,0.033931
2,1990-01-04,cl1s,23.88,23.92,22.83,23.41,50061.0,60995.0,-0.011468
3,1990-01-05,cl1s,23.42,23.70,23.03,23.08,53070.0,57258.0,-0.014197
4,1990-01-08,cl1s,22.60,22.60,21.55,21.62,39720.0,54644.0,-0.065348


In [13]:
# ============================================================
# Core Return and Momentum Features
# ============================================================

MOMENTUM_WINDOWS = [5, 10, 20, 60]

for window in MOMENTUM_WINDOWS:
    
    features_df[f"momentum_{window}d"] = (
        features_df
        .groupby("instrument")["close"]
        .transform(lambda x: np.log(x / x.shift(window)))
    )

# Rolling mean return features
RETURN_MEAN_WINDOWS = [5, 20, 60]

for window in RETURN_MEAN_WINDOWS:
    
    features_df[f"mean_return_{window}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(lambda x: x.rolling(window=window).mean())
    )

display(
    features_df[
        ["date", "instrument", "close", "log_return",
         "momentum_5d", "momentum_20d", "momentum_60d",
         "mean_return_5d", "mean_return_20d"]
    ].head(25)
)

,date,instrument,close,log_return,momentum_5d,momentum_20d,momentum_60d,mean_return_5d,mean_return_20d
0,1990-01-02,cl1s,22.890000,NaN,NaN,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,23.680000,0.033931,NaN,NaN,NaN,NaN,NaN
2,1990-01-04,cl1s,23.410000,-0.011468,NaN,NaN,NaN,NaN,NaN
3,1990-01-05,cl1s,23.080000,-0.014197,NaN,NaN,NaN,NaN,NaN
4,1990-01-08,cl1s,21.620000,-0.065348,NaN,NaN,NaN,NaN,NaN
5,1990-01-09,cl1s,22.070000,0.020600,-0.036481,NaN,NaN,-0.007296,NaN
6,1990-01-10,cl1s,22.900000,0.036918,-0.033494,NaN,NaN,-0.006699,NaN
7,1990-01-11,cl1s,23.140000,0.010426,-0.011601,NaN,NaN,-0.002320,NaN
8,1990-01-12,cl1s,23.130000,-0.000432,0.002164,NaN,NaN,0.000433,NaN
9,1990-01-15,cl1s,22.360000,-0.033857,0.033655,NaN,NaN,0.006731,NaN


252-day rolling z-score of the 20-day mean return for each asset. 

$$Z_t = \frac{R_{20, t} - \mu_{252}(R_{20})}{\sigma_{252}(R_{20})}$$

Where:
- $R_{20, t}$ is the current 20-day mean return window.
- $\mu_{252}$ is the rolling mean of that 20-day return over the past 252 days.
- $\sigma_{252}$ is the rolling standard deviation of that 20-day return over the past 252 days.

$Z > +2.0$ or $Z < -2.0$: Indicates that the market is in an extreme tail event (strongly overbought or oversold).

In [14]:
def rolling_zscore(s, w):
    mean = s.rolling(w).mean()
    std = s.rolling(w).std()
    zscore = (s - mean) / std
    return zscore

features_df["ret_20d_zscore"] = rolling_zscore(features_df["mean_return_20d"], 252)
display(features_df[["date", "instrument", "close", "mean_return_20d", "ret_20d_zscore"]])

,date,instrument,close,mean_return_20d,ret_20d_zscore
0,1990-01-02,cl1s,22.890000,NaN,NaN
1,1990-01-03,cl1s,23.680000,NaN,NaN
2,1990-01-04,cl1s,23.410000,NaN,NaN
3,1990-01-05,cl1s,23.080000,NaN,NaN
4,1990-01-08,cl1s,21.620000,NaN,NaN
...,...,...,...,...,...
32609,2022-06-24,rb1s,11.104921,0.002130,-0.189750
32610,2022-06-27,rb1s,10.982176,0.000682,-0.520354
32611,2022-06-28,rb1s,11.288452,0.000506,-0.559316
32612,2022-06-29,rb1s,10.933430,-0.001147,-0.934334


## 2.2 Volatility Features

Volatility is one of the most important variables in financial markets, especially for commodities and futures contracts.

Periods of high volatility often correspond to:
- market stress,
- regime shifts,
- liquidity shocks,
- or macroeconomic uncertainty.

We therefore construct several volatility-related features.

### Rolling Realized Volatility

We first compute rolling realized volatility using the standard deviation of daily log returns:

$$\sigma_t^{(k)} = \sqrt{\frac{1}{k-1}\sum_{i=0}^{k-1}(r_{t-i} - \bar r_t)^2}$$

over multiple horizons.

### EWMA Volatility

We also compute exponentially weighted volatility estimates, which react faster to recent market shocks.

### Volatility Ratios

Finally, we compute volatility ratios such as:

$$\frac{\sigma_{20}}{\sigma_{60}}$$

to detect volatility regime changes and volatility expansions.

In [15]:
# ============================================================
# Rolling Realized Volatility
# ============================================================

VOL_WINDOWS = [5, 20, 60]

for window in VOL_WINDOWS:

    features_df[f"realized_vol_{window}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(lambda x: x.rolling(window=window).std())
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "log_return",
            "realized_vol_5d",
            "realized_vol_20d",
            "realized_vol_60d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

,date,instrument,log_return,realized_vol_5d,realized_vol_20d,realized_vol_60d
0,1990-01-02,cl1s,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,0.033931,NaN,NaN,NaN
2,1990-01-04,cl1s,-0.011468,NaN,NaN,NaN
3,1990-01-05,cl1s,-0.014197,NaN,NaN,NaN
4,1990-01-08,cl1s,-0.065348,NaN,NaN,NaN


In [16]:
# ============================================================
# EWMA Volatility
# ============================================================

EWMA_SPANS = [10, 20]

for span in EWMA_SPANS:

    features_df[f"ewma_vol_{span}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(
            lambda x: x.ewm(span=span, adjust=False).std()
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "ewma_vol_10d",
            "ewma_vol_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

,date,instrument,ewma_vol_10d,ewma_vol_20d
0,1990-01-02,cl1s,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN
2,1990-01-04,cl1s,0.032101,0.032101
3,1990-01-05,cl1s,0.031324,0.032253
4,1990-01-08,cl1s,0.047855,0.048410


In [17]:
# ============================================================
# Volatility Regime Features
# ============================================================

features_df["vol_ratio_20_60"] = (
    features_df["realized_vol_20d"]
    /
    features_df["realized_vol_60d"]
)

display(
    features_df[
        [
            "date",
            "instrument",
            "realized_vol_20d",
            "realized_vol_60d",
            "vol_ratio_20_60"
        ]
    ].head() #change to head(30) if you want to see more rows
)

,date,instrument,realized_vol_20d,realized_vol_60d,vol_ratio_20_60
0,1990-01-02,cl1s,NaN,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN,NaN


Traditional close-to-close rolling standard deviation is notoriously noisy and inefficient. To provide our metamodel with a cleaner understanding of market structure and regime shifts, we extract several advanced efficiency estimators:

- Garman-Klass Volatility

A highly efficient estimator utilizing Open ($O$), High ($H$), Low ($L$), and Close ($C$) prices. Because it assumes continuous trading, it strictly measures intraday trading volatility and ignores overnight gaps. The daily variance is calculated as:

$$\sigma_{GK}^2 = \frac{1}{2} \left( \ln \frac{H_t}{L_t} \right)^2 - (2 \ln 2 - 1) \left( \ln \frac{C_t}{O_t} \right)^2$$

We take the rolling average of this daily variance over a window of $n$ days, and return the square root.

- Yang-Zhang Volatility

Considered the optimal historical estimator. It achieves minimum estimation error by accounting for both continuous intraday trend drift and overnight price gaps. This is our primary measure of total market risk. It is calculated as the weighted sum of overnight volatility, open-to-close volatility, and Rogers-Satchell (RS) volatility:

$$\sigma_{YZ}^2 = \sigma_{overnight}^2 + k \cdot \sigma_{open\_to\_close}^2 + (1-k) \cdot \sigma_{RS}^2$$

Where the individual components are defined as the rolling variances of the following:
* **Overnight jump:** $\ln \left( \frac{O_t}{C_{t-1}} \right)$
* **Intraday drift:** $\ln \left( \frac{C_t}{O_t} \right)$
* **Rogers-Satchell (trend efficiency):** $\ln \left( \frac{H_t}{O_t} \right) \ln \left( \frac{H_t}{C_t} \right) + \ln \left( \frac{L_t}{O_t} \right) \ln \left( \frac{L_t}{C_t} \right)$
* **Weighting factor $k$:** $k = \frac{0.34}{1.34 + \frac{n+1}{n-1}}$

We complement these with two further range-based estimators that isolate specific components of the price path:

- **Parkinson volatility.** Uses only the high-low range, ignoring open and close entirely:

$$\sigma_{P}^2 = \frac{1}{4 \ln 2} \left( \ln \frac{H_t}{L_t} \right)^2$$

Because it depends purely on the intraday range, it is roughly five times more efficient than close-to-close volatility for capturing the *magnitude of intraday swings*, but by construction it cannot see gaps or drift. It is therefore a clean measure of pure intraday dispersion.

- **Rogers-Satchell volatility (standalone).** The same RS term that enters Yang-Zhang, but extracted as its own feature:

$$\sigma_{RS}^2 = \ln \frac{H_t}{C_t} \ln \frac{H_t}{O_t} + \ln \frac{L_t}{C_t} \ln \frac{L_t}{O_t}$$

Its defining property is that it is **drift-independent**: it stays unbiased even when the price is strongly trending within the day, unlike Parkinson or Garman-Klass. Exposing it separately lets the metamodel contrast a drift-robust volatility estimate against the range-only Parkinson estimate — a divergence between the two is itself informative about whether recent moves were trending or choppy.

In [18]:
# ============================================================
# More efficient volatility calculations then the standard one
# ============================================================

def garman_klass_vol(o, h, l, c, window: int = 20) -> pd.Series:
    hl = 0.5 * np.log(h / l) ** 2
    co = (2 * np.log(2) - 1) * np.log(c / o) ** 2
    daily_var = (hl - co).clip(lower=0)             # ← clip before rolling
    return np.sqrt(daily_var.rolling(window).mean())

def yang_zhang_vol(open, high, low, close, window=20):
    close_prev = close.shift(1)
    overnight = (np.log(open/close_prev)) ** 2
    open_to_close = (np.log(close/open)) ** 2

    sigma_overnight = overnight.rolling(window).mean()
    sigma_open_to_close = open_to_close.rolling(window).mean()

    rs = np.log(high / open) * np.log(high / close) + np.log(low / open) * np.log(low / close)
    sigma_rs = rs.rolling(window).mean()

    k= 0.34 / (1.34 + (window + 1) / (window - 1))
    var = sigma_overnight + k * sigma_open_to_close + (1-k) * sigma_rs
    return np.sqrt(var.clip(lower=0))

open = features_df["open"]
high = features_df["high"]
low = features_df["low"]
close = features_df["close"]

features_df["garman_klass_vol"] = garman_klass_vol(open, high, low, close)
features_df["yang_zhang_vol"] = yang_zhang_vol(open, high, low, close)

display(
    features_df[
        [
            "date",
            "instrument", 
            "garman_klass_vol",
            "yang_zhang_vol"
        ]
    ].iloc[20:] # skip first 20 rows because rolling window starts at 20
)


,date,instrument,garman_klass_vol,yang_zhang_vol
20,1990-01-30,cl1s,0.017687,0.020721
21,1990-01-31,cl1s,0.017293,0.020418
22,1990-02-01,cl1s,0.016092,0.019155
23,1990-02-02,cl1s,0.015685,0.018807
24,1990-02-05,cl1s,0.015379,0.017909
...,...,...,...,...
32609,2022-06-24,rb1s,0.030438,0.031332
32610,2022-06-27,rb1s,0.030484,0.031377
32611,2022-06-28,rb1s,0.030608,0.031597
32612,2022-06-29,rb1s,0.031169,0.031991


### 2.2b Higher Moments and Range Position

The volatility features above capture the **second moment** (dispersion) of returns. Two further dimensions of the return distribution carry independent information:

- **Skew** (third moment) — asymmetry. Negative skew means crashes are sharper than rallies (canonical for equities and crude); positive skew means rallies dominate. Different from vol because it tells us *which side* of zero the dispersion sits on.
- **Kurtosis** (fourth moment) — tail thickness. High kurtosis means jumps are more likely than a normal distribution would predict. Useful complement to vol: an instrument with low vol and high kurtosis is *jumpier* than its dispersion suggests.

We also add features capturing where the current price sits within its recent range, which is **distinct from volatility** because it tracks levels rather than dispersion:

- **`price_range_position_60d`** — position of the current close within the 60-day high-low range. 0 = at the low, 1 = at the high, 0.5 = middle. Differs from Bollinger position because it uses actual extremes, not statistical bands.
- **`range_position_5d_chg`** — 5-day change in range position. Captures the *velocity* of price discovery within the range.

Two final features encode dynamic risk relationships:

- **`downside_vol_20d`** — standard deviation of negative returns only. The Sortino-ratio denominator. Captures *bad-side* volatility, which is the risk metric that actually matters for evaluating long positions.
- **`return_vol_correl_60d`** — 60-day correlation between log returns and changes in volatility. Strongly negative values indicate the canonical "leverage effect" (drops drive vol spikes). Near-zero or positive values flag unusual regimes (e.g., vol expansion without directional bias).

Together these capture seven dimensions of return-distribution shape and price-level positioning that the second-moment vol features above do not.

In [19]:
# ============================================================
# Higher Moments and Range Position Features
# ============================================================

# ------------------------------------------------------------
# Rolling skewness of log returns (20-day and 60-day)
# ------------------------------------------------------------

features_df["skew_20d"] = (
    features_df
    .groupby("instrument")["log_return"]
    .transform(lambda x: x.rolling(20).skew())
)

features_df["skew_60d"] = (
    features_df
    .groupby("instrument")["log_return"]
    .transform(lambda x: x.rolling(60).skew())
)

# ------------------------------------------------------------
# Rolling kurtosis of log returns (20-day)
# ------------------------------------------------------------

features_df["kurt_20d"] = (
    features_df
    .groupby("instrument")["log_return"]
    .transform(lambda x: x.rolling(20).kurt())
)

# ------------------------------------------------------------
# Downside volatility (std of negative returns only, 20-day window)
# ------------------------------------------------------------

def _downside_std(x):
    neg = x[x < 0]
    return np.std(neg) if len(neg) > 1 else np.nan

features_df["downside_vol_20d"] = (
    features_df
    .groupby("instrument")["log_return"]
    .transform(
        lambda x: x.rolling(20).apply(_downside_std, raw=True)
    )
)

# ------------------------------------------------------------
# Price range position over 60-day window
# 0.0 = at 60d low, 1.0 = at 60d high
# ------------------------------------------------------------

hh_60 = (
    features_df
    .groupby("instrument")["high"]
    .transform(lambda x: x.rolling(60).max())
)
ll_60 = (
    features_df
    .groupby("instrument")["low"]
    .transform(lambda x: x.rolling(60).min())
)

features_df["price_range_position_60d"] = (
    (features_df["close"] - ll_60) / (hh_60 - ll_60).replace(0, np.nan)
)

# ------------------------------------------------------------
# 5-day change in range position (velocity)
# ------------------------------------------------------------

features_df["range_position_5d_chg"] = (
    features_df
    .groupby("instrument")["price_range_position_60d"]
    .transform(lambda x: x.diff(5))
)

# ------------------------------------------------------------
# Return / vol-change correlation (60-day, leverage effect)
# ------------------------------------------------------------

# Use realized_vol_20d (already built in section 2.2) as the vol measure
features_df["vol_change_20d"] = (
    features_df
    .groupby("instrument")["realized_vol_20d"]
    .transform(lambda x: x.diff())
)

features_df["return_vol_correl_60d"] = (
    features_df
    .groupby("instrument", group_keys=False)
    .apply(
        lambda g: g["log_return"].rolling(60).corr(g["vol_change_20d"])
    )
)

# Drop the temporary vol_change column
features_df = features_df.drop(columns=["vol_change_20d"])

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    features_df[
        [
            "date",
            "instrument",
            "skew_20d",
            "kurt_20d",
            "downside_vol_20d",
            "price_range_position_60d",
            "range_position_5d_chg",
            "return_vol_correl_60d",
        ]
    ]
    .dropna()
    .head(10)
)

# Quick sanity check on the new features
print("\nSummary statistics for new features (CL only):")
display(
    features_df[features_df["instrument"] == "cl1s"][
        [
            "skew_20d", "skew_60d", "kurt_20d",
            "downside_vol_20d",
            "price_range_position_60d", "range_position_5d_chg",
            "return_vol_correl_60d",
        ]
    ]
    .describe()
    .round(4)
)

/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/3154942715.py:91: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,date,instrument,skew_20d,kurt_20d,downside_vol_20d,price_range_position_60d,range_position_5d_chg,return_vol_correl_60d
80,1990-04-26,cl1s,0.491646,0.754646,0.011521,0.200249,-0.054360,0.127698
81,1990-04-27,cl1s,0.438325,0.626164,0.011253,0.209762,-0.038052,0.147360
82,1990-04-30,cl1s,0.524821,1.008089,0.011253,0.209762,-0.072027,0.146079
83,1990-05-01,cl1s,0.504909,0.827439,0.011253,0.246455,-0.025821,0.152989
84,1990-05-02,cl1s,0.489644,0.822022,0.011377,0.240526,0.014456,0.038866
85,1990-05-03,cl1s,0.377751,0.580028,0.012427,0.145401,-0.054848,0.007817
86,1990-05-04,cl1s,0.253956,0.602608,0.013304,0.143938,-0.065824,-0.002147
87,1990-05-07,cl1s,0.066541,0.261164,0.013827,0.186380,-0.023382,0.009854
88,1990-05-08,cl1s,0.044919,0.858538,0.012911,0.187843,-0.058612,0.005309
89,1990-05-09,cl1s,0.410756,0.531043,0.010238,0.291754,0.051228,0.024177



Summary statistics for new features (CL only):


,skew_20d,skew_60d,kurt_20d,downside_vol_20d,price_range_position_60d,range_position_5d_chg,return_vol_correl_60d
count,8151.0000,8111.0000,8151.0000,8151.0000,8112.0000,8107.0000,8091.0000
mean,-0.1260,-0.1864,0.4673,0.0129,0.5369,0.0003,-0.0657
std,0.6692,0.6802,1.6287,0.0117,0.3143,0.1731,0.2276
min,-3.3786,-4.8544,-1.6205,0.0015,0.0016,-0.8165,-0.8869
25%,-0.5002,-0.4376,-0.5229,0.0077,0.2497,-0.0945,-0.2127
50%,-0.1066,-0.1612,0.0299,0.0106,0.5602,0.0042,-0.0725
75%,0.2631,0.1241,0.9067,0.0145,0.8356,0.1002,0.0826
max,3.1023,2.7558,13.4210,0.1968,1.0000,0.7440,0.6385


In [20]:
def parkinson_vol(h, l, window=20):
    return np.sqrt((np.log(h / l) ** 2 / (4 * np.log(2))).rolling(window).mean())

def rogers_satchell_vol(o, h, l, c, window=20):
    rs = np.log(h / c) * np.log(h / o) + np.log(l / c) * np.log(l / o)
    return np.sqrt(rs.clip(lower=0).rolling(window).mean())

features_df['vol_parkinson_20d'] = features_df.groupby("instrument", group_keys=False).apply(
    lambda g: parkinson_vol(g['high'], g['low']))
features_df['vol_rogers_satchell_20d'] = features_df.groupby("instrument", group_keys=False).apply(
    lambda g: rogers_satchell_vol(g['open'], g['high'], g['low'], g['close']))

/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/2145604809.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df['vol_parkinson_20d'] = features_df.groupby("instrument", group_keys=False).apply(
/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/2145604809.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df['vol_rogers_satchell_20d'] = features_df.groupby("instrum

## 2.3 Volume and Open Interest Features

Futures markets contain additional information beyond prices alone.

In particular:
- trading volume,
- and open interest

can provide insight into:
- market participation,
- liquidity conditions,
- trend confirmation,
- and abnormal trading activity.

We therefore construct several features based on:
- volume dynamics,
- abnormal volume detection,
- and changes in open interest.

### Volume Z-Scores

We compute rolling z-scores of volume:

$$z_t^{vol} = \frac{V_t - \mu_t^{(k)}}{\sigma_t^{(k)}}$$

to detect unusually high or low trading activity.

### Open Interest Changes

We also compute changes in open interest, which may help identify:
- trend strengthening,
- position buildup,
- or liquidation regimes.

### Volume-to-Open-Interest Ratio

Finally, we add **vol_oi_ratio** $= V_t / OI_t$, the ratio of daily volume to open interest. This is a turnover-intensity measure: it tells us how much of the outstanding position base is being churned on a given day. A high ratio flags days dominated by short-term, fast-money flow (intraday speculation, roll activity) relative to the standing book of longer-term positions, which often coincides with less persistent, more reversal-prone moves. It is distinct from the raw volume z-score because it is normalised by committed capital rather than by its own recent history.

All features are computed independently for each instrument.

In [21]:
# ============================================================
# Volume Features
# ============================================================

# Log volume change
features_df["log_volume_change"] = (
    features_df
    .groupby("instrument")["volume"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

# Rolling volume averages
VOLUME_WINDOWS = [5, 20]

for window in VOLUME_WINDOWS:

    features_df[f"volume_mean_{window}d"] = (
        features_df
        .groupby("instrument")["volume"]
        .transform(lambda x: x.rolling(window).mean())
    )

    features_df[f"volume_std_{window}d"] = (
        features_df
        .groupby("instrument")["volume"]
        .transform(lambda x: x.rolling(window).std())
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "volume",
            "log_volume_change",
            "volume_mean_20d",
            "volume_std_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,date,instrument,volume,log_volume_change,volume_mean_20d,volume_std_20d
0,1990-01-02,cl1s,22868.0,NaN,NaN,NaN
1,1990-01-03,cl1s,45177.0,0.680850,NaN,NaN
2,1990-01-04,cl1s,50061.0,0.102654,NaN,NaN
3,1990-01-05,cl1s,53070.0,0.058370,NaN,NaN
4,1990-01-08,cl1s,39720.0,-0.289757,NaN,NaN


In [22]:
# ============================================================
# Volume Z-Score Features
# ============================================================

features_df["volume_zscore_20d"] = (
    (
        features_df["volume"]
        - features_df["volume_mean_20d"]
    )
    /
    features_df["volume_std_20d"]
)

display(
    features_df[
        [
            "date",
            "instrument",
            "volume",
            "volume_mean_20d",
            "volume_std_20d",
            "volume_zscore_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

,date,instrument,volume,volume_mean_20d,volume_std_20d,volume_zscore_20d
0,1990-01-02,cl1s,22868.0,NaN,NaN,NaN
1,1990-01-03,cl1s,45177.0,NaN,NaN,NaN
2,1990-01-04,cl1s,50061.0,NaN,NaN,NaN
3,1990-01-05,cl1s,53070.0,NaN,NaN,NaN
4,1990-01-08,cl1s,39720.0,NaN,NaN,NaN


In [23]:
# ============================================================
# Open Interest Features
# ============================================================

# Log change in open interest
features_df["log_oi_change"] = (
    features_df
    .groupby("instrument")["open_interest"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

# Open interest momentum
OI_WINDOWS = [5, 20]

for window in OI_WINDOWS:

    features_df[f"oi_momentum_{window}d"] = (
        features_df
        .groupby("instrument")["open_interest"]
        .transform(lambda x: np.log(x / x.shift(window)))
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "open_interest",
            "log_oi_change",
            "oi_momentum_5d",
            "oi_momentum_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,date,instrument,open_interest,log_oi_change,oi_momentum_5d,oi_momentum_20d
0,1990-01-02,cl1s,66308.0,NaN,NaN,NaN
1,1990-01-03,cl1s,61428.0,-0.076445,NaN,NaN
2,1990-01-04,cl1s,60995.0,-0.007074,NaN,NaN
3,1990-01-05,cl1s,57258.0,-0.063225,NaN,NaN
4,1990-01-08,cl1s,54644.0,-0.046728,NaN,NaN


### 2.3b Microstructure and Liquidity Features

The volume features above describe **how much** trading is happening. We now add features that describe **how the price moves relative to that volume** — i.e., proxies for trading-cost-related illiquidity that cannot be observed directly from daily OHLCV.

- **`amihud_20d`** — Amihud (2002) illiquidity ratio: the 20-day average of |daily return| divided by dollar volume. Higher values indicate a market where smaller trades move prices more — i.e., less liquid. Widely used in academic asset-pricing studies.

- **`roll_spread_20d`** — Roll (1984) effective bid-ask spread estimator: $2 \sqrt{-\text{Cov}(\Delta p_t, \Delta p_{t-1})}$, derived from the autocovariance of price changes. Negative autocovariance from bid-ask bounce reveals the implicit spread. Only meaningful when the autocovariance is negative; clipped to zero otherwise.

- **`dollar_volume_log`** — log of close × volume. A simple absolute-liquidity-level feature: how much money is changing hands. Useful as a regime indicator (low-liquidity periods behave differently than high-liquidity ones).

- **`kyle_lambda_20d`** — Kyle (1985) price-impact coefficient: regression coefficient of |return| on √volume over a 20-day window. Captures the per-unit-volume price impact: higher λ means the market is more "thin" and a unit of volume moves the price more.

We also add three **single-bar OHLC spread features** that describe the *shape* of each day's price action — pure-microstructure descriptors that require no rolling window:

- **`hl_spread`** $= (H_t - L_t)/C_t$ — the intraday range normalised by the close. A direct, contemporaneous read on how wide the day's trading was; spikes mark stress or news days.
- **`oc_spread`** $= (C_t - O_t)/O_t$ — the open-to-close body of the bar. The signed return realised *during* the session, separating intraday drift from the overnight gap embedded in the close-to-close return.
- **`close_position`** $= (C_t - L_t)/(H_t - L_t)$ — where the close lands within the day's range. Values near 1 mean the market closed on its highs (buyers in control into the close), near 0 means closing on the lows. This is a daily-bar analogue of the stochastic oscillator and a short-horizon read on end-of-day conviction.

These features inform the metamodel about *trading-cost regime* and *bar shape*, which are structurally distinct from price-volatility regime: a market can be quiet (low vol) but illiquid (high Amihud) or volatile (high vol) but liquid (low Amihud).

In [24]:
# ============================================================
# Microstructure and Liquidity Features
# ============================================================

# ------------------------------------------------------------
# Helper: dollar volume series (used by multiple features)
# ------------------------------------------------------------

features_df["dollar_volume"] = (
    features_df["close"] * features_df["volume"]
)

# ------------------------------------------------------------
# Amihud illiquidity (20-day rolling mean of |return| / dollar volume)
# ------------------------------------------------------------

features_df["amihud_daily"] = (
    features_df["log_return"].abs()
    / features_df["dollar_volume"].replace(0, np.nan)
)

features_df["amihud_20d"] = (
    features_df
    .groupby("instrument")["amihud_daily"]
    .transform(lambda x: x.rolling(20).mean())
)

# ------------------------------------------------------------
# Roll's effective spread (20-day window)
#   spread ≈ 2 * sqrt( -Cov(Δp_t, Δp_{t-1}) )
# Only defined where the autocovariance is negative (bid-ask bounce).
# Clipped to zero otherwise.
# ------------------------------------------------------------

features_df["price_change"] = (
    features_df
    .groupby("instrument")["close"]
    .transform(lambda x: x.diff())
)

# Lagged price change for autocovariance computation
features_df["price_change_lag1"] = (
    features_df
    .groupby("instrument")["price_change"]
    .transform(lambda x: x.shift(1))
)

# Rolling mean of the product (this is the autocovariance of price changes)
features_df["cov_pricechanges_20d"] = (
    features_df
    .groupby("instrument", group_keys=False)
    .apply(
        lambda g: (g["price_change"] * g["price_change_lag1"]).rolling(20).mean()
    )
)

features_df["roll_spread_20d"] = (
    2 * np.sqrt((-features_df["cov_pricechanges_20d"]).clip(lower=0))
)

# ------------------------------------------------------------
# Log dollar volume (liquidity-level feature)
# ------------------------------------------------------------

features_df["dollar_volume_log"] = (
    np.log(features_df["dollar_volume"].replace(0, np.nan))
)

# ------------------------------------------------------------
# Kyle's lambda: price impact per unit of √volume (20-day window)
#   lambda = Cov(|return|, sqrt(volume)) / Var(sqrt(volume))
# ------------------------------------------------------------

features_df["abs_return"] = features_df["log_return"].abs()
features_df["sqrt_volume"] = np.sqrt(features_df["volume"].replace(0, np.nan))

def _kyle_lambda(g, w=20):
    abs_r = g["abs_return"]
    sqrt_v = g["sqrt_volume"]
    cov = (abs_r * sqrt_v).rolling(w).mean() \
          - abs_r.rolling(w).mean() * sqrt_v.rolling(w).mean()
    var = sqrt_v.rolling(w).var()
    return cov / var.replace(0, np.nan)

features_df["kyle_lambda_20d"] = (
    features_df
    .groupby("instrument", group_keys=False)
    .apply(_kyle_lambda)
)

# ------------------------------------------------------------
# Clean up temporary columns
# ------------------------------------------------------------

features_df = features_df.drop(
    columns=[
        "dollar_volume",
        "amihud_daily",
        "price_change",
        "price_change_lag1",
        "cov_pricechanges_20d",
        "abs_return",
        "sqrt_volume",
    ]
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    features_df[
        [
            "date",
            "instrument",
            "amihud_20d",
            "roll_spread_20d",
            "dollar_volume_log",
            "kyle_lambda_20d",
        ]
    ]
    .dropna()
    .head(10)
)

print("\nSummary statistics (CL only):")
display(
    features_df[features_df["instrument"] == "cl1s"][
        [
            "amihud_20d",
            "roll_spread_20d",
            "dollar_volume_log",
            "kyle_lambda_20d",
        ]
    ]
    .describe()
    .round(6)
)

/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/4112946061.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/4112946061.py:88: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_kyle_lambda)


,date,instrument,amihud_20d,roll_spread_20d,dollar_volume_log,kyle_lambda_20d
21,1990-01-31,cl1s,2.211039e-08,0.208648,13.601472,0.000045
22,1990-02-01,cl1s,2.166982e-08,0.000000,13.715945,0.000072
23,1990-02-02,cl1s,2.208714e-08,0.127751,13.461878,0.000118
24,1990-02-05,cl1s,2.031203e-08,0.396221,13.435080,0.000021
25,1990-02-06,cl1s,1.971069e-08,0.205445,13.601254,0.000002
26,1990-02-07,cl1s,1.807312e-08,0.349169,13.398205,0.000013
27,1990-02-08,cl1s,1.808994e-08,0.390069,13.626928,0.000015
28,1990-02-09,cl1s,1.918403e-08,0.366033,13.464512,-0.000075
29,1990-02-12,cl1s,1.736098e-08,0.397278,13.700351,-0.000020
30,1990-02-13,cl1s,1.624097e-08,0.318490,13.340279,0.000045



Summary statistics (CL only):


,amihud_20d,roll_spread_20d,dollar_volume_log,kyle_lambda_20d
count,7744.0,8150.000000,8131.000000,7744.000000
mean,0.0,0.437357,15.382938,0.000014
std,0.0,0.781829,1.661884,0.000121
min,0.0,0.000000,6.569131,-0.001029
25%,0.0,0.000000,13.761699,-0.000029
50%,0.0,0.024670,15.206729,0.000025
75%,0.0,0.565949,17.012119,0.000064
max,0.0,6.314644,18.297034,0.000918


In [25]:
features_df['hl_spread']      = (features_df['high'] - features_df['low']) / features_df['close']
features_df['oc_spread']      = (features_df['close'] - features_df['open']) / features_df['open']
features_df['close_position'] = (features_df['close'] - features_df['low']) / (features_df['high'] - features_df['low'])

## 2.4 Technical Indicators

We now construct a set of classical technical indicators.

These indicators are widely used in systematic trading because they summarize different aspects of recent price dynamics:

- **RSI** captures overbought and oversold conditions.
- **Bollinger z-scores** measure how far the current price is from its recent moving average.
- **MACD** captures trend strength through the difference between short-term and long-term exponential moving averages.

To broaden coverage across the standard families of oscillators and volume-flow indicators, we add the following (computed per instrument via TA-Lib):

- **`bb_width`** $= (\text{Upper} - \text{Lower})/\text{Mid}$ and **`bb_position`** $= (C_t - \text{Lower})/(\text{Upper} - \text{Lower})$ — derived directly from the 20-day Bollinger Bands. `bb_width` is a *volatility-squeeze* indicator: narrow bands precede expansions; `bb_position` locates the close within the bands (complementary to the Bollinger z-score, which is unbounded, whereas this is naturally framed in [0, 1]).
- **`willr_14` — Williams %R.** A momentum oscillator bounded in [-100, 0] measuring the close relative to the 14-day high-low range; an inverted, unsmoothed cousin of the stochastic that reacts quickly to overbought/oversold extremes.
- **`stoch_k`, `stoch_d` — Stochastic oscillator.** %K positions the close within the recent range and %D is its moving-average smoothing. Crossovers and extremes flag momentum exhaustion.
- **`obv` — On-Balance Volume.** A running total of volume signed by the day's price direction. It is a *volume-based trend-confirmation* measure: price moves backed by rising OBV are better supported than moves on flat or falling OBV.
- **`mfi_14` — Money Flow Index.** A volume-weighted RSI: it incorporates typical price *and* volume to gauge buying versus selling pressure, so it can diverge from price-only RSI when a move is not backed by volume.
- **`atr_14` — Average True Range.** The 14-day average of the true range (which includes overnight gaps). Unlike the return-based volatility estimators in §2.2, ATR is an *absolute, price-level* measure of typical daily movement, useful for contextualising the size of recent moves.

Although these indicators are individually simple, they span the trend, momentum-oscillator, and volume-flow families, giving the metamodel interpretable signals about the market environment in which the primary signal is more or less reliable.

In [26]:
# ============================================================
# RSI Indicator
# ============================================================

def compute_rsi(series, window=14):
    """
    Compute the Relative Strength Index (RSI).
    """
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi


features_df["rsi_14d"] = (
    features_df
    .groupby("instrument")["close"]
    .transform(lambda x: compute_rsi(x, window=14))
)

display(
    features_df[
        ["date", "instrument", "close", "rsi_14d"]
    ].head(70)
)

,date,instrument,close,rsi_14d
0,1990-01-02,cl1s,22.890000,NaN
1,1990-01-03,cl1s,23.680000,NaN
2,1990-01-04,cl1s,23.410000,NaN
3,1990-01-05,cl1s,23.080000,NaN
4,1990-01-08,cl1s,21.620000,NaN
...,...,...,...,...
65,1990-04-04,cl1s,20.535266,36.024845
66,1990-04-05,cl1s,20.171901,35.474006
67,1990-04-06,cl1s,19.881210,37.060703
68,1990-04-09,cl1s,19.144100,31.780822


In [27]:
# ============================================================
# Bollinger-Style Price Z-Score
# ============================================================

BOLLINGER_WINDOWS = [20, 60]

for window in BOLLINGER_WINDOWS:

    rolling_mean = (
        features_df
        .groupby("instrument")["close"]
        .transform(lambda x: x.rolling(window=window).mean())
    )

    rolling_std = (
        features_df
        .groupby("instrument")["close"]
        .transform(lambda x: x.rolling(window=window).std())
    )

    features_df[f"price_zscore_{window}d"] = (
        (features_df["close"] - rolling_mean) / rolling_std
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "close",
            "price_zscore_20d",
            "price_zscore_60d"
        ]
    ].head(70)
)

,date,instrument,close,price_zscore_20d,price_zscore_60d
0,1990-01-02,cl1s,22.890000,NaN,NaN
1,1990-01-03,cl1s,23.680000,NaN,NaN
2,1990-01-04,cl1s,23.410000,NaN,NaN
3,1990-01-05,cl1s,23.080000,NaN,NaN
4,1990-01-08,cl1s,21.620000,NaN,NaN
...,...,...,...,...,...
65,1990-04-04,cl1s,20.535266,-1.594795,-1.792779
66,1990-04-05,cl1s,20.171901,-2.371197,-2.030857
67,1990-04-06,cl1s,19.881210,-2.535736,-2.170688
68,1990-04-09,cl1s,19.144100,-2.979594,-2.622275


In [28]:
# ============================================================
# MACD Indicator
# ============================================================

def compute_macd(series, short_span=12, long_span=26, signal_span=9):
    """
    Compute MACD, MACD signal line, and MACD histogram.
    """
    ema_short = series.ewm(span=short_span, adjust=False).mean()
    ema_long = series.ewm(span=long_span, adjust=False).mean()

    macd = ema_short - ema_long
    macd_signal = macd.ewm(span=signal_span, adjust=False).mean()
    macd_hist = macd - macd_signal

    return macd, macd_signal, macd_hist


macd_results = (
    features_df
    .groupby("instrument")["close"]
    .apply(lambda x: pd.DataFrame({
        "macd": compute_macd(x)[0],
        "macd_signal": compute_macd(x)[1],
        "macd_hist": compute_macd(x)[2],
    }, index=x.index))
)

macd_results = macd_results.reset_index(level=0, drop=True)

features_df[["macd", "macd_signal", "macd_hist"]] = macd_results[
    ["macd", "macd_signal", "macd_hist"]
]

display(
    features_df[
        [
            "date",
            "instrument",
            "close",
            "macd",
            "macd_signal",
            "macd_hist"
        ]
    ].head(70)
)

,date,instrument,close,macd,macd_signal,macd_hist
0,1990-01-02,cl1s,22.890000,0.000000,0.000000,0.000000
1,1990-01-03,cl1s,23.680000,0.063020,0.012604,0.050416
2,1990-01-04,cl1s,23.410000,0.090138,0.028111,0.062027
3,1990-01-05,cl1s,23.080000,0.084032,0.039295,0.044737
4,1990-01-08,cl1s,21.620000,-0.038176,0.023801,-0.061977
...,...,...,...,...,...,...
65,1990-04-04,cl1s,20.535266,-0.379125,-0.424619,0.045493
66,1990-04-05,cl1s,20.171901,-0.419359,-0.423567,0.004207
67,1990-04-06,cl1s,19.881210,-0.469292,-0.432712,-0.036580
68,1990-04-09,cl1s,19.144100,-0.561865,-0.458543,-0.103323


- Average Directional Index (ADX 14)

ADX measures the absolute strength of a trend, completely independent of its direction. It is derived from the smoothed averages of True Range ($TR$) and Directional Movement ($+DM$ and $-DM$). The core calculation for the Directional Index ($DX$) before the final 14-day smoothing is:

$$DX = 100 \times \frac{|+DI - -DI|}{|+DI + -DI|}$$

- Distance from 200d MA in Sigma Units 

Rather than calculating the raw percentage difference between the close price and the 200-day Simple Moving Average (SMA), we normalize this macroeconomic distance using the localized rolling 60-day standard deviation ($\sigma_{60}$).

$$Distance_{\sigma} = \frac{C_t - SMA_{200}(C)}{\sigma_{60}(C)}$$

In [29]:
#ATX 14 - average directional index, measures strength of trend not direction

open = features_df["open"]
high = features_df["high"]
low = features_df["low"]
close = features_df["close"]

tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)

atr = tr.ewm(alpha=1/14, adjust=False).mean()
up_move = high - high.shift(1)
down_move = low.shift(1) - low
plus_dm = up_move.where((up_move > down_move) & (up_move > 0), 0.0)
minus_dm = down_move.where((down_move > up_move) & (down_move > 0), 0.0)
plus_di = 100 * pd.Series(plus_dm, index=features_df.index).ewm(alpha=1/14, adjust=False).mean() / atr
minus_di = 100 * pd.Series(minus_dm, index=features_df.index).ewm(alpha=1/14, adjust=False).mean() / atr
dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)

features_df["atx_14"] = dx.ewm(alpha=1/14, adjust=False).mean()

ma_200 = close.rolling(200).mean()
std_60 = close.rolling(60).std()
features_df["distance_from_200d_ma"] = (close - ma_200) / std_60.replace(0, np.nan)

display(features_df[
    [
        "date", 
        "instrument",
        "high",
        "low",
        "close",
        "atx_14",
        "distance_from_200d_ma"
    ]
].iloc[200:])


,date,instrument,high,low,close,atx_14,distance_from_200d_ma
200,1990-10-16,cl1s,34.871074,33.216933,34.772719,23.904929,2.304066
201,1990-10-17,cl1s,34.819145,33.379953,33.379953,22.300326,2.072833
202,1990-10-18,cl1s,33.240677,31.522932,32.878558,21.880765,2.010175
203,1990-10-19,cl1s,31.662209,31.021537,31.021537,21.743920,1.702999
204,1990-10-22,cl1s,28.236005,28.236005,28.236005,22.772300,1.206338
...,...,...,...,...,...,...,...
32609,2022-06-24,rb1s,11.126945,10.645361,11.104921,20.464032,2.745495
32610,2022-06-27,rb1s,11.243817,10.909057,10.982176,20.049497,2.630628
32611,2022-06-28,rb1s,11.305483,10.851209,11.288452,19.461737,2.897622
32612,2022-06-29,rb1s,11.355991,10.732281,10.933430,19.224087,2.604690


In [30]:
def add_talib_block(g):
    h, l, c, v = g['high'].values, g['low'].values, g['close'].values, g['volume'].values.astype(float)
    g['willr_14'] = talib.WILLR(h, l, c, timeperiod=14)
    g['stoch_k'], g['stoch_d'] = talib.STOCH(h, l, c)
    g['obv'] = talib.OBV(c, v)
    g['mfi_14'] = talib.MFI(h, l, c, v, timeperiod=14)
    g['atr_14'] = talib.ATR(h, l, c, timeperiod=14)
    u, m, lo = talib.BBANDS(c, timeperiod=20)
    g['bb_width'] = (u - lo) / m
    g['bb_position'] = (c - lo) / (u - lo)
    g['vol_oi_ratio'] = g['volume'] / g['open_interest'].replace(0, np.nan)
    return g

features_df = features_df.groupby("instrument", group_keys=False).apply(add_talib_block)

/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/1359470586.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df = features_df.groupby("instrument", group_keys=False).apply(add_talib_block)


## 2.5 Time-Series Dependence Features

Financial time series are not independent through time.

In particular:
- returns may exhibit short-term momentum or mean reversion,
- volatility tends to cluster,
- and trends may persist for several consecutive periods.

We therefore construct features designed to capture temporal dependence structures.

### Rolling Return Autocorrelation

We compute rolling autocorrelation of returns to measure:
- short-term trend persistence,
- or mean-reversion effects.

### Absolute Return Autocorrelation

We also compute autocorrelation of absolute returns:

$$
Corr(|r_t|, |r_{t-1}|)
$$

which is a classical proxy for volatility clustering.

### Trend Persistence

Finally, we compute rolling proportions of positive-return days in order to measure the persistence of directional trends.

In [31]:
# ============================================================
# Rolling Return Autocorrelation
# ============================================================

AUTOCORR_WINDOWS = [20, 60]

for window in AUTOCORR_WINDOWS:

    features_df[f"autocorr_return_{window}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(
            lambda x: x.rolling(window).corr(x.shift(1))
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "log_return",
            "autocorr_return_20d",
            "autocorr_return_60d"
        ]
    ].head(80)
)

,date,instrument,log_return,autocorr_return_20d,autocorr_return_60d
0,1990-01-02,cl1s,NaN,NaN,NaN
1,1990-01-03,cl1s,0.033931,NaN,NaN
2,1990-01-04,cl1s,-0.011468,NaN,NaN
3,1990-01-05,cl1s,-0.014197,NaN,NaN
4,1990-01-08,cl1s,-0.065348,NaN,NaN
...,...,...,...,...,...
75,1990-04-19,cl1s,0.042214,0.113603,0.034054
76,1990-04-20,cl1s,-0.002653,0.099340,0.033998
77,1990-04-23,cl1s,0.013196,0.084618,0.029485
78,1990-04-24,cl1s,-0.003677,0.061846,-0.010933


In [32]:
# ============================================================
# Absolute Return Autocorrelation
# ============================================================

features_df["abs_log_return"] = (
    features_df["log_return"].abs()
)

for window in AUTOCORR_WINDOWS:

    features_df[f"autocorr_abs_return_{window}d"] = (
        features_df
        .groupby("instrument")["abs_log_return"]
        .transform(
            lambda x: x.rolling(window).corr(x.shift(1))
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "abs_log_return",
            "autocorr_abs_return_20d",
            "autocorr_abs_return_60d"
        ]
    ].head(80)
)

,date,instrument,abs_log_return,autocorr_abs_return_20d,autocorr_abs_return_60d
0,1990-01-02,cl1s,NaN,NaN,NaN
1,1990-01-03,cl1s,0.033931,NaN,NaN
2,1990-01-04,cl1s,0.011468,NaN,NaN
3,1990-01-05,cl1s,0.014197,NaN,NaN
4,1990-01-08,cl1s,0.065348,NaN,NaN
...,...,...,...,...,...
75,1990-04-19,cl1s,0.042214,0.322005,0.141686
76,1990-04-20,cl1s,0.002653,0.131837,0.057133
77,1990-04-23,cl1s,0.013196,0.161377,0.101448
78,1990-04-24,cl1s,0.003677,0.187625,0.099012


In [33]:
# ============================================================
# Trend Persistence Features
# ============================================================

# Positive-return indicator
features_df["positive_return"] = (
    (features_df["log_return"] > 0).astype(int)
)

PERSISTENCE_WINDOWS = [10, 20]

for window in PERSISTENCE_WINDOWS:

    features_df[f"positive_return_ratio_{window}d"] = (
        features_df
        .groupby("instrument")["positive_return"]
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "positive_return",
            "positive_return_ratio_10d",
            "positive_return_ratio_20d"
        ]
    ].head(70)
)

,date,instrument,positive_return,positive_return_ratio_10d,positive_return_ratio_20d
0,1990-01-02,cl1s,0,NaN,NaN
1,1990-01-03,cl1s,1,NaN,NaN
2,1990-01-04,cl1s,0,NaN,NaN
3,1990-01-05,cl1s,0,NaN,NaN
4,1990-01-08,cl1s,0,NaN,NaN
...,...,...,...,...,...
65,1990-04-04,cl1s,0,0.6,0.4
66,1990-04-05,cl1s,0,0.5,0.4
67,1990-04-06,cl1s,0,0.4,0.4
68,1990-04-09,cl1s,0,0.3,0.4


### 2.5b Spectral and Fractal Features

Beyond direct autocorrelations, we capture structural properties of the price-return process:

- **Dominant cycle period** (FFT-based): the period length at which the most spectral power is concentrated, after detrending. Identifies dominant oscillatory horizons in the price series.
- **Spectral entropy**: how uniformly spectral power is distributed across frequencies. Low entropy = a few frequencies dominate (structured signal); high entropy = energy spread evenly (noise-like).
- **Hurst exponent (90-day, R/S method)**: persistence measure. H ≈ 0.5 = random walk, H > 0.5 = trending, H < 0.5 = mean-reverting.
- **DFA exponent (Detrended Fluctuation Analysis)**: similar to Hurst but more robust to non-stationarities.
- **Approximate entropy**: regularity/complexity of the return series. Lower values indicate more predictable patterns.

We extend this block with two **memory / stationarity** features and two **information-theoretic** features drawn from the López de Prado framework:

- **`close_fracdiff` — Fractional differentiation (d ≈ 0.4).** Standard differencing (i.e. taking returns) achieves stationarity but destroys the long memory in the price level. Fractional differentiation applies a fractional-order difference that makes the series approximately stationary *while retaining the maximum amount of memory*. This gives the metamodel a near-stationary level feature — something that ordinary returns cannot provide, since they discard the slow-moving information about where price sits relative to its own history.
- **`sadf` — Supremum Augmented Dickey-Fuller statistic.** A rolling, right-tailed unit-root test that detects *explosive* (bubble-like) behaviour rather than mean reversion. High SADF values flag periods where the price is in an explosive regime; these are precisely the environments where trend-following primary signals are most fragile and most prone to abrupt reversal, so this is a direct regime descriptor for the metamodel.
- **`shannon_entropy_hist_20d` — Histogram Shannon entropy.** The Shannon entropy of the binned 20-day return *distribution*. It measures the disorder of recent return values: low entropy means returns are concentrated (a tight, predictable distribution), high entropy means they are spread out and disordered. This is distinct from spectral entropy (frequency domain) and approximate entropy (sequence regularity) — it operates on the *value distribution* of returns.
- **`lz_complexity_20d` — Lempel-Ziv complexity.** The compressibility of the binarised return-sign sequence over 20 days. It counts the number of distinct patterns required to reconstruct the sequence: a repetitive, structured sign pattern compresses well (low complexity), while an erratic one does not (high complexity). It complements the entropy measures with a *sequence-pattern* view of predictability.

These features are computationally heavier than rolling autocorrelations but capture different structural information. Note that `sadf` and `close_fracdiff` carry long warm-up periods, which is worth keeping in mind for the missing-data filter in Phase 4.

In [34]:
def rolling_apply_array(s, w, fn):
    vals = s.values 
    n = len(s)
    out_vals = np.full(n, np.nan)
    for i in range(w - 1, n):
        window = vals[i - w + 1 : i + 1]
        if np.all(np.isfinite(window)):
            try:
                out_vals[i] = fn(window)
            except Exception:
                out_vals[i] = np.nan
    return pd.Series(out_vals, index=s.index)

def dominant_cycle_period(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if len(x) < 8 or np.std(x) == 0:
        return np.nan
    
    t = np.arange(len(x))
    coeffs = np.polyfit(t, x, 1)
    detrended = x - np.polyval(coeffs, t)
    
    spec = np.abs(np.fft.rfft(detrended))
    # Zero out: DC + the lowest 2 bins (which absorb residual trend curvature)
    spec[:3] = 0
    
    if spec.sum() == 0:
        return np.nan
    
    k = np.argmax(spec)
    return len(x) / max(k, 1)

def spectral_entropy(x):
    spec = np.abs(np.fft.rfft(x - x.mean())) ** 2
    spec = spec[1:]  # drop DC component
    total = spec.sum()
    
    if total <= 0:
        return np.nan
    
    p = spec / total
    p = p[p > 0]
    return -(p * np.log(p)).sum() / np.log(len(p))

def hurst(x: np.ndarray) -> float:
    """
    Hurst exponent via the rescaled range (R/S) method.
    Original Hurst-Mandelbrot estimator.
    H ≈ 0.5: random walk
    H > 0.5: persistent (trending)
    H < 0.5: anti-persistent (mean-reverting)
    """
    x = np.asarray(x, dtype=float)
    N = len(x)
    if N < 20 or np.std(x) == 0:
        return np.nan
    
    # Work on increments (returns), not levels
    returns = np.diff(x)
    if len(returns) < 10 or np.std(returns) == 0:
        return np.nan
    
    # Build R/S for several scales
    scales = []
    rs_values = []
    for n in [10, 20, 30, 40, 50, 60, 80]:
        if n > len(returns):
            break
        # Split into chunks of size n
        n_chunks = len(returns) // n
        if n_chunks < 2:
            continue
        rs_chunk = []
        for i in range(n_chunks):
            chunk = returns[i * n : (i + 1) * n]
            mean_c = chunk.mean()
            z = np.cumsum(chunk - mean_c)
            R = z.max() - z.min()         # range of cumulative deviations
            S = np.std(chunk, ddof=1)     # std of chunk
            if S > 0 and np.isfinite(R / S):
                rs_chunk.append(R / S)
        if len(rs_chunk) >= 1:
            scales.append(n)
            rs_values.append(np.mean(rs_chunk))
    
    if len(scales) < 3:
        return np.nan
    
    # log(R/S) ~ H * log(n) for fBm
    slope, _ = np.polyfit(np.log(scales), np.log(rs_values), 1)
    return slope

def dfa(x):
    if len(x) < 16:
        return np.nan
    
    y = np.cumsum(x - x.mean())
    
    scales = [s for s in [4, 8, 16] if s < len(x) // 2]
    if len(scales) < 2:
        return np.nan
    
    f = []
    for s in scales:
        n_segments = len(y) // s
        rms_list = []
        for i in range(n_segments):
            seg = y[i * s : (i + 1) * s]
            t = np.arange(s)
            poly = np.polyfit(t, seg, 1)
            trend = np.polyval(poly, t)
            rms_list.append(np.sqrt(np.mean((seg - trend) ** 2)))
        f.append(np.mean(rms_list) if rms_list else np.nan)
    
    f = np.array(f)
    if np.any(~np.isfinite(f)) or np.any(f <= 0):
        return np.nan
    
    slope, _ = np.polyfit(np.log(scales), np.log(f), 1)
    return slope

def approx_entropy(x: np.ndarray, m: int = 2, r_mult: float = 0.2) -> float:
    """
    Approximate entropy (Pincus 1991). Lower = more predictable.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < m + 2:
        return np.nan
    
    sd = np.std(x)
    if sd == 0 or not np.isfinite(sd):
        return np.nan
    r = r_mult * sd
    
    def _phi(m_):
        # Build all sub-sequences of length m_
        N = n - m_ + 1
        if N <= 0:
            return np.nan
        patterns = np.zeros((N, m_))
        for i in range(N):
            patterns[i] = x[i : i + m_]
        
        # For each pattern, count how many others are within r in Chebyshev distance
        counts = np.zeros(N)
        for i in range(N):
            diff = np.abs(patterns - patterns[i])
            max_diff = diff.max(axis=1)
            counts[i] = np.sum(max_diff <= r) / N
        
        # Avoid log(0)
        counts = np.where(counts > 0, counts, 1e-12)
        return np.mean(np.log(counts))
    
    phi_m = _phi(m)
    phi_m1 = _phi(m + 1)
    
    if not (np.isfinite(phi_m) and np.isfinite(phi_m1)):
        return np.nan
    return phi_m - phi_m1


In [35]:
LOG_CLOSE_COL = "log_close"
features_df[LOG_CLOSE_COL] = np.log(features_df["close"])

# Hurst — applied to log prices, 90-day window
features_df["hurst_90d"] = (
    features_df
    .groupby("instrument")[LOG_CLOSE_COL]
    .transform(lambda x: x.rolling(90).apply(hurst, raw=True))
)

# DFA alpha — applied to log returns, 90-day window
features_df["dfa_alpha_90d"] = (
    features_df
    .groupby("instrument")["log_return"]
    .transform(lambda x: x.rolling(90).apply(dfa, raw=True))
)

# Dominant cycle period — applied to log prices, 60-day window
features_df["dominant_cycle_period"] = (
    features_df
    .groupby("instrument")[LOG_CLOSE_COL]
    .transform(lambda x: x.rolling(60).apply(dominant_cycle_period, raw=True))
)

# Spectral entropy — applied to log returns, 60-day window
features_df["spectral_entropy"] = (
    features_df
    .groupby("instrument")["log_return"]
    .transform(lambda x: x.rolling(60).apply(spectral_entropy, raw=True))
)

# Approximate entropy — applied to log returns, 20-day window
features_df["approx_entropy_20d"] = (
    features_df
    .groupby("instrument")["log_return"]
    .transform(lambda x: x.rolling(20).apply(approx_entropy, raw=True))
)

# Clean up the temporary log_close column
features_df = features_df.drop(columns=[LOG_CLOSE_COL])

display(
    features_df[
        [
            "date", "instrument",
            "hurst_90d", "dfa_alpha_90d",
            "dominant_cycle_period", "spectral_entropy",
            "approx_entropy_20d",
        ]
    ].dropna().head()
)


,date,instrument,hurst_90d,dfa_alpha_90d,dominant_cycle_period,spectral_entropy,approx_entropy_20d
90,1990-05-10,cl1s,0.562468,0.706787,20.0,0.878509,0.115344
91,1990-05-11,cl1s,0.589131,0.521639,20.0,0.885936,0.115344
92,1990-05-14,cl1s,0.609911,0.560662,15.0,0.894262,0.188307
93,1990-05-15,cl1s,0.568424,0.555204,15.0,0.895918,0.261270
94,1990-05-16,cl1s,0.591625,0.645504,15.0,0.903247,0.261270


In [36]:
def lempel_ziv_complexity(seq):
    i, c, u, v, v_max, n = 0, 1, 1, 1, 1, len(seq)
    if n == 0: return 0
    while u + v <= n:
        if seq[i + v - 1] == seq[u + v - 1]:
            v += 1
        else:
            v_max = max(v_max, v); i += 1
            if i == u:
                c += 1; u += v_max; v = v_max = 1; i = 0
            else:
                v = 1
    return c + (v != 1)

def frac_diff_ffd(series, d, thres=1e-5):
    w = [1.0]; k = 1
    while True:
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < thres: break
        w.append(w_k); k += 1
    w = np.array(w[::-1]); width = len(w) - 1
    out = pd.Series(index=series.index, dtype=float)
    for i in range(width, len(series)):
        out.iloc[i] = np.dot(w, series.iloc[i - width:i + 1].values)
    return out

def sadf_rolling(close, step=5):
    sadf_vals = np.full(len(close), np.nan)
    for i in range(25, len(close), step):
        window = close.iloc[max(0, i - 252):i].values
        try:
            sadf_vals[i] = ts.adfuller(window, maxlag=1, autolag=None)[0]
        except Exception:
            pass
    out = pd.Series(sadf_vals, index=close.index)
    return out.ffill()

def shannon_entropy_vectorized(series, window, bins=10):
    arr = series.values
    result = np.full(len(arr), np.nan)
    for i in range(window - 1, len(arr)):
        x = arr[i - window + 1:i + 1]
        x = x[~np.isnan(x)]
        if len(x) == 0: continue
        hist, _ = np.histogram(x, bins=bins)
        p = hist[hist > 0].astype(float); p /= p.sum()
        result[i] = -np.sum(p * np.log2(p))
    return pd.Series(result, index=series.index)

def lz_rolling_fast(series, window):
    binary = (series.values > 0).astype(int)
    result = np.full(len(binary), np.nan)
    for i in range(window - 1, len(binary)):
        seq = ''.join(map(str, binary[i - window + 1:i + 1]))
        result[i] = lempel_ziv_complexity(seq)
    return pd.Series(result, index=series.index)


features_df['close_fracdiff'] = features_df.groupby("instrument")['close'].transform(
    lambda x: frac_diff_ffd(x, d=0.4))
features_df['sadf']= sadf_rolling(features_df['close'], step=5)  
features_df['shannon_entropy_hist_20d'] = features_df.groupby("instrument")['log_return'].transform(
    lambda x: shannon_entropy_vectorized(x, 20))
features_df['shannon_entropy_hist_60d'] = features_df.groupby("instrument")['log_return'].transform(
    lambda x: shannon_entropy_vectorized(x, 60))
features_df['lz_complexity_20d'] = features_df.groupby("instrument")['log_return'].transform(
    lambda x: lz_rolling_fast(x, 20))
features_df['lz_complexity_60d'] = features_df.groupby("instrument")['log_return'].transform(
    lambda x: lz_rolling_fast(x, 60))

## 2.6 Cross-Sectional Features

So far, all features were computed independently for each instrument through time.

We now introduce **cross-sectional features**, which compare instruments within the Energy asset class at a given date.

These features attempt to capture:
- relative strength,
- relative volatility,
- sector dispersion,
- and intra-sector positioning.

Cross-sectional information is particularly important in multi-asset systematic trading because the attractiveness of a signal may depend not only on the state of one asset, but also on its position relative to the rest of the sector.

All cross-sectional features are computed date-by-date across the Energy universe.

In [37]:
# ============================================================
# Cross-Sectional Momentum Rank
# ============================================================

features_df["momentum_rank_20d"] = (
    features_df
    .groupby("date")["momentum_20d"]
    .rank(pct=True)
)

features_df["momentum_rank_60d"] = (
    features_df
    .groupby("date")["momentum_60d"]
    .rank(pct=True)
)

display(
    features_df[
        [
            "date",
            "instrument",
            "momentum_20d",
            "momentum_rank_20d",
            "momentum_60d",
            "momentum_rank_60d"
        ]
    ]
    .sort_values(["date", "momentum_rank_20d"])
    .head(70)
)

,date,instrument,momentum_20d,momentum_rank_20d,momentum_60d,momentum_rank_60d
0,1990-01-02,cl1s,NaN,NaN,NaN,NaN
8171,1990-01-02,ho1s,NaN,NaN,NaN,NaN
24444,1990-01-02,rb1s,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN,NaN
8172,1990-01-03,ho1s,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
21,1990-01-31,cl1s,0.004042,1.000000,NaN,NaN
8193,1990-02-01,ho1s,-0.216586,0.333333,NaN,NaN
24466,1990-02-01,rb1s,-0.056451,0.666667,NaN,NaN
22,1990-02-01,cl1s,0.016391,1.000000,NaN,NaN


In [38]:
# ============================================================
# Relative Volatility Features
# ============================================================

# Sector average volatility by date
sector_avg_vol_20d = (
    features_df
    .groupby("date")["realized_vol_20d"]
    .transform("mean")
)

sector_avg_vol_60d = (
    features_df
    .groupby("date")["realized_vol_60d"]
    .transform("mean")
)

# Relative volatility
features_df["relative_vol_20d"] = (
    features_df["realized_vol_20d"]
    / sector_avg_vol_20d
)

features_df["relative_vol_60d"] = (
    features_df["realized_vol_60d"]
    / sector_avg_vol_60d
)

display(
    features_df[
        [
            "date",
            "instrument",
            "realized_vol_20d",
            "relative_vol_20d",
            "realized_vol_60d",
            "relative_vol_60d"
        ]
    ].head(70)
)

,date,instrument,realized_vol_20d,relative_vol_20d,realized_vol_60d,relative_vol_60d
0,1990-01-02,cl1s,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
65,1990-04-04,cl1s,0.012938,0.923443,0.015101,0.802494
66,1990-04-05,cl1s,0.013313,0.923523,0.014396,0.795834
67,1990-04-06,cl1s,0.013197,0.953786,0.014388,0.804626
68,1990-04-09,cl1s,0.015195,1.030745,0.015087,0.830741


In [39]:
# ============================================================
# Sector Dispersion Features
# ============================================================

# Cross-sectional std of momentum across assets
features_df["sector_momentum_dispersion_20d"] = (
    features_df
    .groupby("date")["momentum_20d"]
    .transform("std")
)

features_df["sector_momentum_dispersion_60d"] = (
    features_df
    .groupby("date")["momentum_60d"]
    .transform("std")
)

# Cross-sectional std of volatility across assets
features_df["sector_vol_dispersion_20d"] = (
    features_df
    .groupby("date")["realized_vol_20d"]
    .transform("std")
)

display(
    features_df[
        [
            "date",
            "instrument",
            "sector_momentum_dispersion_20d",
            "sector_momentum_dispersion_60d",
            "sector_vol_dispersion_20d"
        ]
    ].head(70)
)

,date,instrument,sector_momentum_dispersion_20d,sector_momentum_dispersion_60d,sector_vol_dispersion_20d
0,1990-01-02,cl1s,NaN,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN,NaN
...,...,...,...,...,...
65,1990-04-04,cl1s,0.063415,0.028504,0.001094
66,1990-04-05,cl1s,0.062656,0.009315,0.001460
67,1990-04-06,cl1s,0.068239,0.007904,0.001864
68,1990-04-09,cl1s,0.080652,0.009267,0.001917


In [40]:
# ============================================================
# Relative Momentum vs Sector Mean
# ============================================================

sector_mean_momentum_20d = (
    features_df
    .groupby("date")["momentum_20d"]
    .transform("mean")
)

features_df["relative_momentum_20d"] = (
    features_df["momentum_20d"]
    - sector_mean_momentum_20d
)

display(
    features_df[
        [
            "date",
            "instrument",
            "momentum_20d",
            "relative_momentum_20d"
        ]
    ]
    .sort_values(["date", "relative_momentum_20d"])
    .head(70)
)

,date,instrument,momentum_20d,relative_momentum_20d
0,1990-01-02,cl1s,NaN,NaN
8171,1990-01-02,ho1s,NaN,NaN
24444,1990-01-02,rb1s,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN
8172,1990-01-03,ho1s,NaN,NaN
...,...,...,...,...
21,1990-01-31,cl1s,0.004042,0.121233
8193,1990-02-01,ho1s,-0.216586,-0.131037
24466,1990-02-01,rb1s,-0.056451,0.029098
22,1990-02-01,cl1s,0.016391,0.101940


### 2.6b Seasonality Features

The features built so far depend on prices, returns, volatilities, or cross-sectional positioning. None of them capture the **time of year**, yet energy commodities have well-documented seasonal demand patterns:

- **Heating oil** demand peaks in winter (Northern Hemisphere Q4-Q1).
- **Gasoline** (RBOB) demand peaks in summer driving season (April-August).
- **Crude oil** is exposed to summer driving demand and Atlantic hurricane season (June-November).
- **Natural gas** has both winter heating demand and a secondary summer cooling peak.
- **Institutional flows** create end-of-quarter rebalancing effects across the panel.

We construct six smooth seasonal features that encode these patterns without binary cliff-edges:

- **`day_of_year_sin` / `day_of_year_cos`** — cyclic encoding of the position within the calendar year. Distance-preserving (January 1 and December 31 are close in 2D space, unlike with raw day numbers).
- **`heating_season`** — smooth tent function over Oct-Mar peaking in mid-winter. Captures heating demand cycle.
- **`driving_season_progress`** — smooth tent over Apr-Aug peaking in July. Captures summer gasoline demand.
- **`hurricane_season_indicator`** — smooth tent over Jun-Nov peaking in September. Captures Gulf-of-Mexico supply disruption risk.
- **`quarter_progress`** — fraction of the way through the current quarter [0, 1]. Captures institutional rebalancing flows around quarter-end.

These features are identical across instruments at the same date (purely date-derived). Their value is conditional: the metamodel can learn that a primary signal is more or less reliable during specific seasonal windows.

In [41]:
# ============================================================
# Seasonality Features
# ============================================================

def _seasonal_ramp(months, days_in_month, start_month, peak_month, end_month):
    """
    Smooth tent function over a window of months [start_month, end_month],
    peaking at peak_month. Returns 0 outside the window, 1 at the peak.
    Handles year-wrap (e.g., heating: Oct -> Mar) via modulo arithmetic.
    """
    month_frac = (months - 1) + (days_in_month - 1) / 31.0
    
    diff = month_frac - (peak_month - 1)
    diff = (diff + 6) % 12 - 6   # wrap to [-6, 6]
    
    dist_to_start = (peak_month - start_month) % 12
    if dist_to_start == 0:
        dist_to_start = 1
    dist_to_end = (end_month - peak_month) % 12
    if dist_to_end == 0:
        dist_to_end = 1
    
    ramp = np.where(
        diff < 0,
        1 + diff / dist_to_start,
        1 - diff / dist_to_end,
    )
    return np.clip(ramp, 0, 1)


# ------------------------------------------------------------
# Day-of-year cyclic encoding
# ------------------------------------------------------------

doy = features_df["date"].dt.dayofyear

features_df["day_of_year_sin"] = np.sin(2 * np.pi * doy / 365.25)
features_df["day_of_year_cos"] = np.cos(2 * np.pi * doy / 365.25)

# ------------------------------------------------------------
# Physical-demand seasonal ramps
# ------------------------------------------------------------

months = features_df["date"].dt.month.values
days_in_month = features_df["date"].dt.days_in_month.values

features_df["heating_season"] = _seasonal_ramp(
    months, days_in_month,
    start_month=10, peak_month=1, end_month=4,
)

features_df["driving_season_progress"] = _seasonal_ramp(
    months, days_in_month,
    start_month=4, peak_month=7, end_month=8,
)

features_df["hurricane_season_indicator"] = _seasonal_ramp(
    months, days_in_month,
    start_month=6, peak_month=9, end_month=11,
)

# ------------------------------------------------------------
# Quarter progress (fraction through current quarter)
# ------------------------------------------------------------

quarter_starts = features_df["date"].dt.to_period("Q").dt.start_time
quarter_ends = features_df["date"].dt.to_period("Q").dt.end_time

quarter_length = (quarter_ends - quarter_starts).dt.days.clip(lower=1)
elapsed = (features_df["date"] - quarter_starts).dt.days

features_df["quarter_progress"] = elapsed / quarter_length

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    features_df[
        [
            "date",
            "instrument",
            "day_of_year_sin",
            "day_of_year_cos",
            "heating_season",
            "driving_season_progress",
            "hurricane_season_indicator",
            "quarter_progress",
        ]
    ]
    .head(10)
)

# Quick sanity check: monthly mean values
print("\nMonthly mean values of seasonal indicators (CL only):")
display(
    features_df[features_df["instrument"] == "cl1s"]
    .assign(month=features_df["date"].dt.month)
    .groupby("month")[
        ["heating_season", "driving_season_progress", "hurricane_season_indicator"]
    ]
    .mean()
    .round(3)
)

,date,instrument,day_of_year_sin,day_of_year_cos,heating_season,driving_season_progress,hurricane_season_indicator,quarter_progress
0,1990-01-02,cl1s,0.034398,0.999408,0.677419,0.0,0.0,0.011236
1,1990-01-03,cl1s,0.051584,0.998669,0.677419,0.0,0.0,0.022472
2,1990-01-04,cl1s,0.068755,0.997634,0.677419,0.0,0.0,0.033708
3,1990-01-05,cl1s,0.085906,0.996303,0.677419,0.0,0.0,0.044944
4,1990-01-08,cl1s,0.137185,0.990545,0.677419,0.0,0.0,0.078652
5,1990-01-09,cl1s,0.154204,0.988039,0.677419,0.0,0.0,0.089888
6,1990-01-10,cl1s,0.171177,0.985240,0.677419,0.0,0.0,0.101124
7,1990-01-11,cl1s,0.188099,0.982150,0.677419,0.0,0.0,0.112360
8,1990-01-12,cl1s,0.204966,0.978769,0.677419,0.0,0.0,0.123596
9,1990-01-15,cl1s,0.255182,0.966893,0.677419,0.0,0.0,0.157303



Monthly mean values of seasonal indicators (CL only):


,heating_season,driving_season_progress,hurricane_season_indicator
month,,,
1,0.677,0.000,0.000
2,0.374,0.000,0.000
3,0.011,0.000,0.000
4,0.000,0.312,0.000
5,0.000,0.656,0.000
6,0.000,0.978,0.312
7,0.000,0.032,0.656
8,0.000,0.000,0.989
9,0.000,0.000,0.532


## 2.7 Inter-energy spreads

**Goal.** Capture the *internal economic state of the energy complex* — refining margins and crude-vs-natural-gas relative pricing — that no single-instrument OHLCV feature can express.

The 3-2-1 crack spread is the canonical refining margin: when it widens, refiners are more profitable and bid up crude. The CL/NG ratio is a textbook mean-reverting pair anchored by shale economics. Their z-scores and short-term changes turn these levels into regime indicators the metamodel can use to distinguish "tight refining" from "oversupplied" states.

These features are the same for every energy instrument on a given date — they describe the complex, not the individual contract — so we merge on `date` only.

In [42]:
# 3-2-1 crack, oil/gas ratio, sub-cracks, plus z-scores and changes.
# These features are the SAME for every energy instrument — they describe the 
# state of the energy complex. 

# Finding cl, ho, rb, ng in the ohlcv DataFrame
cl_close = ohlcv[ohlcv["instrument"] == "cl1s"].set_index("date")["close"]
ho_close = ohlcv[ohlcv["instrument"] == "ho1s"].set_index("date")["close"]
rb_close = ohlcv[ohlcv["instrument"] == "rb1s"].set_index("date")["close"]
ng_close = ohlcv[ohlcv["instrument"] == "ng1s"].set_index("date")["close"]

# Finding crack_321, oil/gas ratio, and their z-scores and changes
crack_321 = 3 * rb_close + 2 * ho_close - 3 * cl_close
crack_321_zscore_252d = (crack_321 - crack_321.rolling(window=252).mean()) / crack_321.rolling(window=252).std()

crack_321_change_5d = crack_321.diff(5)

oil_gas_ratio = cl_close / ng_close
oil_gas_ratio_zscore_252d = (oil_gas_ratio - oil_gas_ratio.rolling(window=252).mean()) / oil_gas_ratio.rolling(window=252).std()

In [43]:
# Joining the metrics with the ohlcv_energy DataFrame
new_features = pd.DataFrame({
    "crack_321": crack_321,
    "crack_321_zscore_252d": crack_321_zscore_252d,
    "crack_321_change_5d": crack_321_change_5d,
    "oil_gas_ratio": oil_gas_ratio,
    "oil_gas_ratio_zscore_252d": oil_gas_ratio_zscore_252d
}).reset_index()

features_df = features_df.merge(new_features, on="date", how="left")

In [44]:
features_df[
        [
            "date",
            "instrument",
            "crack_321",
            "crack_321_zscore_252d",
            "crack_321_change_5d",
            "oil_gas_ratio",
            "oil_gas_ratio_zscore_252d"
        ]
    ]

,date,instrument,crack_321,crack_321_zscore_252d,crack_321_change_5d,oil_gas_ratio,oil_gas_ratio_zscore_252d
0,1990-01-02,cl1s,-65.151500,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,-67.459100,NaN,NaN,NaN,NaN
2,1990-01-04,cl1s,-66.807500,NaN,NaN,NaN,NaN
3,1990-01-05,cl1s,-66.016800,NaN,NaN,NaN,NaN
4,1990-01-08,cl1s,-61.819000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
32609,2022-06-24,rb1s,-63.851050,-1.124888,6.500901,9683.340910,0.532665
32610,2022-06-27,rb1s,-66.369053,-1.350527,-1.294619,9459.676033,0.389352
32611,2022-06-28,rb1s,-67.602141,-1.453229,-1.077951,9613.511153,0.491329
32612,2022-06-29,rb1s,-67.033870,-1.387633,-4.194278,9547.825085,0.449403


## 2.8 Cross-Asset Regime

**Goal.** Build a *risk-on / risk-off* indicator using only the cross-asset data already in our universe — no external feeds required.

Equities (ES, NQ, FESX) describe the global appetite for risk; gold and copper describe defensive vs growth-oriented flows. Their average returns and volatilities form a composite picture of the macro state. The risk-on score (equity − gold + copper) compresses this into a single number that tells the metamodel whether the primary signal is firing into a benign or stressed macro backdrop.

In [45]:
# Finding the close price series for ES, NQ, FESX, GC, and HG
es_close = ohlcv[ohlcv["instrument"] == "es1s"].set_index("date")["close"]
nq_close = ohlcv[ohlcv["instrument"] == "nq1s"].set_index("date")["close"]
fesx_close = ohlcv[ohlcv["instrument"] == "fesx1s"].set_index("date")["close"]
gc_close = ohlcv[ohlcv["instrument"] == "gc1s"].set_index("date")["close"]
hg_close = ohlcv[ohlcv["instrument"] == "hg1s"].set_index("date")["close"]

# Calculate log returns for each instrument
es_ret = np.log(es_close).diff()
nq_ret = np.log(nq_close).diff()
fesx_ret = np.log(fesx_close).diff()
gold_ret = np.log(gc_close).diff()
copper_ret = np.log(hg_close).diff()

In [46]:
# Group the equity returns into a DataFrame for easier calculations
equity_ret_df = pd.DataFrame({
    "es1s": es_ret,
    "nq1s": nq_ret,
    "fesx1s": fesx_ret
})

# Calculate the average return across all equity instruments
equity_ret = equity_ret_df.mean(axis=1)

# Calculate the 5-day rolling sum of equity returns
equity_avg_ret_5d = equity_ret.rolling(5).sum()

# Volatility features: rolling 20-day volatility of equity returns, then average across instruments, then z-score vs 252-day history
es_vol   = es_ret.rolling(20).std()   * np.sqrt(252)
nq_vol   = nq_ret.rolling(20).std()   * np.sqrt(252)
fesx_vol = fesx_ret.rolling(20).std() * np.sqrt(252)

equity_avg_vol_20d = pd.concat([es_vol, nq_vol, fesx_vol], axis=1).mean(axis=1)
equity_vol_zscore_252d = (
    equity_avg_vol_20d - equity_avg_vol_20d.rolling(252).mean()
) / equity_avg_vol_20d.rolling(252).std()

# Risk-on score (sum of 5-day returns of equities minus gold plus copper)
risk_on_score = equity_ret.rolling(5).sum() - gold_ret.rolling(5).sum() + copper_ret.rolling(5).sum()

In [47]:
# Join all these cross-asset metrics into a single DataFrame, then merge with features_df
cross_asset_metrics_df = pd.DataFrame({
    "equity_avg_ret_5d": equity_avg_ret_5d,
    "equity_avg_vol_20d": equity_avg_vol_20d,
    "equity_vol_zscore_252d": equity_vol_zscore_252d,
    "risk_on_score": risk_on_score
}).reset_index()

# Joining the cross-asset metrics with the features_df
features_df = features_df.merge(cross_asset_metrics_df, on="date", how="left")

In [48]:
features_df[['date', 'instrument', 'equity_avg_ret_5d', 'equity_avg_vol_20d', 'equity_vol_zscore_252d', 'risk_on_score']]

,date,instrument,equity_avg_ret_5d,equity_avg_vol_20d,equity_vol_zscore_252d,risk_on_score
0,1990-01-02,cl1s,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
32609,2022-06-24,rb1s,0.060446,0.325411,1.457285,-0.022620
32610,2022-06-27,rb1s,0.049427,0.317074,1.347363,-0.006671
32611,2022-06-28,rb1s,0.013270,0.310472,1.259044,-0.044454
32612,2022-06-29,rb1s,0.013048,0.310891,1.253651,-0.018927


## 2.9 Metals Ratios

**Goal.** Use the classic *copper/gold* and *gold/silver* pairs as proxies for the growth-vs-fear and industrial-vs-monetary regimes.

Copper proxies industrial demand; gold proxies fear. A rising copper/gold ratio signals pro-growth conditions historically bullish for oil demand. The gold/silver ratio captures a complementary dimension — monetary stress vs industrial activity. Together they let the metamodel condition on a richer macro picture than any single equity index can provide.

In [49]:
# Getting the close price series for HG, GC, and SI
hg_close = ohlcv[ohlcv["instrument"] == "hg1s"].set_index("date")["close"]
gc_close = ohlcv[ohlcv["instrument"] == "gc1s"].set_index("date")["close"]
si_close = ohlcv[ohlcv["instrument"] == "si1s"].set_index("date")["close"]

# Calculate the ratio Cobre / Oro
copper_gold_ratio = hg_close / gc_close

# Z-score for the copper/gold ratio vs its own 252-day history
copper_gold_ratio_zscore_252d = (copper_gold_ratio - copper_gold_ratio.rolling(window=252).mean()) / copper_gold_ratio.rolling(window=252).std()

# 20-day percentage change in the copper/gold ratio
copper_gold_ratio_chg_20d = copper_gold_ratio.pct_change(20)

#  Calculate the ratio Gold / Silver
gold_silver_ratio = gc_close / si_close

/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/1618725054.py:13: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  copper_gold_ratio_chg_20d = copper_gold_ratio.pct_change(20)


In [50]:
# Join all these metal metrics into a single DataFrame, then merge with features_df
metals_metrics_df = pd.DataFrame({
    "copper_gold_ratio": copper_gold_ratio,
    "copper_gold_ratio_zscore_252d": copper_gold_ratio_zscore_252d,
    "copper_gold_ratio_chg_20d": copper_gold_ratio_chg_20d,
    "gold_silver_ratio": gold_silver_ratio
}).reset_index()

# Joining the metals metrics with the features_df
features_df = features_df.merge(metals_metrics_df, on="date", how="left")

In [51]:
features_df[["date", "instrument", "copper_gold_ratio", "copper_gold_ratio_zscore_252d", "copper_gold_ratio_chg_20d", "gold_silver_ratio"]]

,date,instrument,copper_gold_ratio,copper_gold_ratio_zscore_252d,copper_gold_ratio_chg_20d,gold_silver_ratio
0,1990-01-02,cl1s,0.002641,NaN,NaN,76.546735
1,1990-01-03,cl1s,0.002766,NaN,NaN,75.938697
2,1990-01-04,cl1s,0.002708,NaN,NaN,75.126523
3,1990-01-05,cl1s,0.002681,NaN,NaN,75.858867
4,1990-01-08,cl1s,0.002749,NaN,NaN,76.271186
...,...,...,...,...,...,...
32609,2022-06-24,rb1s,0.005984,-4.309285,-0.110149,106.299495
32610,2022-06-27,rb1s,0.006037,-3.945425,-0.102614,105.840009
32611,2022-06-28,rb1s,0.006071,-3.687092,-0.105815,107.225390
32612,2022-06-29,rb1s,0.006084,-3.529247,-0.105990,107.698985


## 2.10 Cross-Asset Correlations

**Goal.** Identify *which kind of regime* the primary signal is firing in: is oil currently trading as a growth asset, an inflation hedge, or a pure risk asset?

A trend signal on CL behaves very differently depending on whether oil is moving with copper (growth story), with gold (inflation story), or with equities (risk-on/off story). The 60-day rolling correlations of each instrument's returns with copper, gold, and the S&P give the metamodel a direct read on the dominant driver. When the correlation structure shifts, the reliability of the primary signal often shifts with it.

In [52]:
# Extracting the close price series for HG, GC, and ES to compute their log returns
hg_close = ohlcv[ohlcv["instrument"] == "hg1s"].set_index("date")["close"]
gc_close = ohlcv[ohlcv["instrument"] == "gc1s"].set_index("date")["close"]
es_close = ohlcv[ohlcv["instrument"] == "es1s"].set_index("date")["close"]

hg_ret = np.log(hg_close).diff()
gc_ret = np.log(gc_close).diff()
es_ret = np.log(es_close).diff()

#  List of your 4 target assets
list_metrics = []

# Calculating the rolling correlations for each target asset against HG, GC, and ES
for target_asset in ENERGY_INSTRUMENTS:
    target_close = ohlcv[ohlcv["instrument"] == target_asset].set_index("date")["close"]
    target_ret = np.log(target_close).diff()
    
    # Calculate rolling correlations with a 60-day window
    asset_copper_corr = target_ret.rolling(60).corr(hg_ret)
    asset_gold_corr = target_ret.rolling(60).corr(gc_ret)
    asset_equity_corr = target_ret.rolling(60).corr(es_ret)
    
    # Temporal Dataframe for this asset's correlations
    df_temporal = pd.DataFrame({
        "asset_copper_corr_60d": asset_copper_corr,
        "asset_gold_corr_60d": asset_gold_corr,
        "asset_equity_corr_60d": asset_equity_corr
    }).reset_index()
    
    df_temporal["instrument"] = target_asset
    
    list_metrics.append(df_temporal)

In [53]:
# Concatenating all the correlation DataFrames into a single DataFrame
correlation_df = pd.concat(list_metrics, ignore_index=True)

# Joining the correlation metrics with the features_df
features_df = features_df.merge(correlation_df, on=["date", "instrument"], how="left")

## 2.11 Cross-Asset Momentum

**Goal.** Use *cross-asset confirmation* of the primary signal — particularly copper as a leading indicator — to filter out signals firing against the macro tape.

Copper trends often precede oil trends because both react to industrial demand news, but copper trades more cleanly without OPEC distortions. We use copper's recent return as a leading feature and build a sign-agreement concordance score across copper, equities, and gold momentum against the primary signal's direction. A primary signal firing against three independent cross-asset trends is, ex-ante, a weaker signal than one with macro confirmation.

In [54]:
# 1. Extract close prices and calculate log returns for ALL reference macro assets
hg_close = ohlcv[ohlcv["instrument"] == "hg1s"].set_index("date")["close"]  # Copper
gc_close = ohlcv[ohlcv["instrument"] == "gc1s"].set_index("date")["close"]  # Gold
es_close = ohlcv[ohlcv["instrument"] == "es1s"].set_index("date")["close"]  # S&P 500
nq_close = ohlcv[ohlcv["instrument"] == "nq1s"].set_index("date")["close"]  # Nasdaq
fesx_close = ohlcv[ohlcv["instrument"] == "fesx1s"].set_index("date")["close"]  # Euro Stoxx

hg_ret = np.log(hg_close).diff()
gc_ret = np.log(gc_close).diff()
es_ret = np.log(es_close).diff()
nq_ret = np.log(nq_close).diff()
fesx_ret = np.log(fesx_close).diff()

# 2. Calculate Global Reference Momentums
# For Equities, we combine the 3 indices by taking the daily cross-sectional mean of their returns
eq_returns_df = pd.DataFrame({"es": es_ret, "nq": nq_ret, "fesx": fesx_ret})
eq_mom = eq_returns_df.mean(axis=1).rolling(20).sum()

cu_mom = hg_ret.rolling(20).sum()
au_mom = gc_ret.rolling(20).sum()

# Direct Copper features
copper_mom_20d = cu_mom
copper_lead_5d = hg_ret.rolling(5).sum()

# 3. Process Concordance for each of your target assets (Energy Universe)
asset_mom_list = []
target_assets = ["cl1s", "ho1s", "rb1s", "ng1s"]  # Defined in Phase 1 of your notebook
signals_df = signals.set_index("date")

for target_asset in target_assets:
    # Get the complete timeline in ohlcv for this specific asset
    target_dates = ohlcv[ohlcv["instrument"] == target_asset]["date"].unique()
    
    # Safely reindex the global macro metrics to the asset's timeline
    cu_mom_target = cu_mom.reindex(target_dates)
    au_mom_target = au_mom.reindex(target_dates)
    eq_mom_target = eq_mom.reindex(target_dates)
    cu_mom_20d_target = copper_mom_20d.reindex(target_dates)
    cu_lead_5d_target = copper_lead_5d.reindex(target_dates)
    
    # Extract the asset's signal and align it to the same timeline
    if target_asset in signals_df.columns:
        sig = signals_df[target_asset].reindex(target_dates)
    else:
        sig = pd.Series(np.nan, index=target_dates)
        
    # Replace inactive signals (0) with NaN as required by the original design
    sig_sign = np.sign(sig).replace(0, np.nan)
    
    # Calculate bit-by-bit concordance (range 0 to 3)
    # Note: NaN == anything evaluates to False, protecting the initial rolling windows
    concord = (
        (np.sign(eq_mom_target) == sig_sign).astype(int)
        + (np.sign(cu_mom_target) == sig_sign).astype(int)
        + (np.sign(au_mom_target) == sig_sign).astype(int)
    )
    
    # Apply the .where() mask: the indicator is only valid on days with an active signal (+1 or -1)
    cross_asset_momentum_concordance = concord.where(sig_sign.notna())
    
    # Build the clean DataFrame for the current asset
    temp_df = pd.DataFrame({
        "date": target_dates,
        "copper_mom_20d": cu_mom_20d_target.values,
        "copper_lead_5d": cu_lead_5d_target.values,
        "cross_asset_momentum_concordance": cross_asset_momentum_concordance.values
    })
    temp_df["instrument"] = target_asset
    
    asset_mom_list.append(temp_df)

# 4. Concatenate the results vertically to consolidate the long format
all_mom_metrics_df = pd.concat(asset_mom_list, ignore_index=True)

# 5. Integration via left merge into your general features dataset (features_df)
features_df = features_df.merge(all_mom_metrics_df, on=["date", "instrument"], how="left")

In [55]:
features_df[["date", "instrument", "copper_mom_20d", "copper_lead_5d", "cross_asset_momentum_concordance"]].head(70)

,date,instrument,copper_mom_20d,copper_lead_5d,cross_asset_momentum_concordance
0,1990-01-02,cl1s,NaN,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN,NaN
...,...,...,...,...,...
65,1990-04-04,cl1s,0.048105,0.032097,NaN
66,1990-04-05,cl1s,0.025555,0.008446,NaN
67,1990-04-06,cl1s,0.047346,-0.010678,NaN
68,1990-04-09,cl1s,0.015109,-0.035357,NaN


## 2.12 Cross-Asset Volatility

**Goal.** Distinguish *idiosyncratic* energy stress from *systemic* macro stress.

A vol spike confined to energy (e.g. an OPEC shock) creates different metamodel conditions than a universe-wide vol spike (e.g. a Covid-style risk-off). We compute a 20-day vol per instrument, average across the full universe to get a "GVIX-equivalent" index, z-score it against its one-year history, and form the ratio of energy-average vol to universe-average vol. Trend signals systematically work better in low-vol regimes; this block tells the metamodel which type of regime it's in.

In [56]:
# Universe-wide vol regime + how stressed this instrument is vs the rest.
# 1. Pivot the dataset natively to get a "wide" format (dates as rows, instruments as columns)
px = ohlcv.pivot(index="date", columns="instrument", values="close")

# 2. Calculate daily log returns for ALL instruments simultaneously
ret = np.log(px).diff()

# Define the asset classes (Make sure these match the exact strings in your dataset)
energy_cols = ["cl1s", "ho1s", "rb1s", "ng1s"]
equity_cols = ["es1s", "nq1s", "fesx1s"]

# Get a list of all instruments dynamically from the pivoted DataFrame
universe_cols = list(px.columns)

# 3. Calculate the 20-day rolling volatility for every instrument, annualized
# Pandas applies this column by column automatically
vol_per_instr = ret.rolling(20).std() * np.sqrt(252)

# 4. Calculate the macro volatility metrics using cross-sectional means (axis=1)
multiasset_vol_index = vol_per_instr[universe_cols].mean(axis=1)

# Manual Z-score for the multi-asset volatility index (252-day window)
multiasset_vol_zscore_252d = (
    multiasset_vol_index - multiasset_vol_index.rolling(window=252).mean()
) / multiasset_vol_index.rolling(window=252).std()

equity_vol_avg_20d = vol_per_instr[equity_cols].mean(axis=1)

# How stressed the energy complex is relative to the entire market
relative_energy_vol = vol_per_instr[energy_cols].mean(axis=1) / multiasset_vol_index

In [57]:
# 5. Assemble the final metrics DataFrame
volatility_metrics_df = pd.DataFrame({
    "multiasset_vol_index": multiasset_vol_index,
    "multiasset_vol_zscore_252d": multiasset_vol_zscore_252d,
    "equity_vol_avg_20d": equity_vol_avg_20d,
    "relative_energy_vol": relative_energy_vol
}).reset_index() # Converts the "date" index back into a standard column

# 6. Merge with your main features dataset (features_df)
# Since these are macro indicators (not specific to one instrument), we only merge on "date"
features_df = features_df.merge(volatility_metrics_df, on="date", how="left")

In [58]:
features_df[["date", "instrument", "multiasset_vol_index", "multiasset_vol_zscore_252d", "equity_vol_avg_20d", "relative_energy_vol"]].head(70)

,date,instrument,multiasset_vol_index,multiasset_vol_zscore_252d,equity_vol_avg_20d,relative_energy_vol
0,1990-01-02,cl1s,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
65,1990-04-04,cl1s,0.225092,NaN,NaN,0.988115
66,1990-04-05,cl1s,0.228427,NaN,NaN,1.001785
67,1990-04-06,cl1s,0.226337,NaN,NaN,0.970455
68,1990-04-09,cl1s,0.236365,NaN,NaN,0.990059


## 2.13 Copper Leading Indicators

**Goal.** Reuse the Session 1 *Trend Scanning* methodology on copper rather than the target instrument, treating copper's trend strength as an *exogenous* regime indicator for energy.

Because copper trends often lead oil trends, a statistically significant copper trend (high backward-looking t-value) carries information about the regime under which the primary energy signal is operating. We compute copper's price z-score and the t-statistic of its trend slope over a 30-day window. These features describe the macro tape from a different angle than equities or volatility — copper-specific signal often persists even when other macro factors are noisy.

In [59]:
def calculate_backward_trend_t_value(price_series: pd.Series, window: int = 30) -> pd.Series:
    """
    Calculates the t-value of the slope from an OLS regression of price on time,
    computed over a fixed rolling lookback window.
    """
    # Pre-allocate output series with the same timeline index
    t_values = pd.Series(index=price_series.index, dtype=float)
    p = price_series.values
    
    # Pre-compute parts of X since the time index array [0, 1, 2, ..., window-1] 
    # is identical for every single rolling window
    x = np.arange(window, dtype=float)
    x_mean = x.mean()
    x_dev = x - x_mean
    ssx = (x_dev ** 2).sum()
    df = window - 2
    
    if ssx == 0 or df <= 0:
        return t_values

    # Rolling window loop
    for i in range(window, len(p)):
        y = p[i - window : i]  # Strict past slice of length 'window'
        y_mean = y.mean()
        
        # OLS slope (beta) coefficient calculation
        beta = (x_dev * (y - y_mean)).sum() / ssx
        
        # Intercept (alpha) calculation
        alpha = y_mean - beta * x_mean
        
        # Residuals and Standard Error calculation
        residuals = y - (alpha + beta * x)
        sigma2 = (residuals ** 2).sum() / df
        
        if sigma2 <= 0:
            continue
            
        se_beta = np.sqrt(sigma2 / ssx)
        
        if se_beta == 0:
            continue
            
        # Calculate and store the t-statistic (beta / standard error)
        t_values.iat[i] = beta / se_beta
        
    return t_values

In [60]:
# --- Main Feature Calculation and Integration Pipeline ---

# 1. Isolate the Copper close prices and set the date timeline as index
hg_close = ohlcv[ohlcv["instrument"] == "hg1s"].set_index("date")["close"]

# 2. Calculate the 20-day rolling Z-score manually
copper_zscore_20d = (
    hg_close - hg_close.rolling(window=20).mean()
) / hg_close.rolling(window=20).std()

# 3. Calculate the rolling 30-day OLS trend t-value
copper_trend_t_value = calculate_backward_trend_t_value(hg_close, window=30)

# 4. Construct the clean macro metrics DataFrame
copper_metrics_df = pd.DataFrame({
    "copper_zscore_20d": copper_zscore_20d,
    "copper_trend_t_value": copper_trend_t_value
}).reset_index()  # Convert the 'date' index back into a column

# 5. Integrate into your main features DataFrame
# Since Copper indicators are macro/exogenous regimes, we only merge on "date"
features_df = features_df.merge(copper_metrics_df, on="date", how="left")

In [61]:
features_df[["date", "instrument", "copper_zscore_20d", "copper_trend_t_value"]].head(70)

,date,instrument,copper_zscore_20d,copper_trend_t_value
0,1990-01-02,cl1s,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN
...,...,...,...,...
65,1990-04-04,cl1s,0.736902,7.566141
66,1990-04-05,cl1s,0.251805,7.214083
67,1990-04-06,cl1s,0.860514,6.597435
68,1990-04-09,cl1s,-0.577219,6.280250


## 2.14 Macro Context

**Goal.** Bring in the *external* macro variables that energy traders watch most closely — the dollar, equity vol, and the long end of the curve — using free data from Yahoo Finance.

DXY (dollar strength) mechanically weighs on dollar-denominated commodities; VIX summarises global risk appetite from outside our equity-futures universe; the US 10-year yield captures real-rate and growth-expectation effects. We compute multi-horizon returns and changes plus a rolling 60-day correlation of the energy basket with DXY, giving the metamodel a direct read on the macro environment. This is the only feature block requiring external data; if `yfinance` is unavailable, the rest of the pipeline runs unchanged.

In [62]:
# Pulling DXY, VIX, US 10y yield from Yahoo Finance for the date range in features_df.
# Returns a long-form DataFrame indexed by date that we can use to derive features.
tickers = {"dxy": "DX-Y.NYB", "vix": "^VIX", "us10y": "^TNX"}
frames = []
for name, tkr in tickers.items():
    df = yf.download(
        tkr,
        start=features_df["date"].min(),
        end=features_df["date"].max() + pd.Timedelta(days=1),  # yfinance end is exclusive
        progress=False,
        auto_adjust=False,
    )
    if df.empty:
        print(f"  Warning: no data returned for {tkr}")
        continue
    # yfinance can return a MultiIndex column ('Close','^VIX'); flatten if so
    s = df["Close"]
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    s = s.rename(name)
    frames.append(s)

macro_df = pd.concat(frames, axis=1).sort_index()
macro_df.index = pd.to_datetime(macro_df.index).tz_localize(None)
macro_df = macro_df.reset_index().rename(columns={macro_df.index.name or "Date": "date"})
# Normalise the date column name (yfinance returns 'Date' with capital D)
macro_df.columns = ["date" if c.lower() == "date" else c for c in macro_df.columns]

macro_df.head()

,index,dxy,vix,us10y
0,1990-01-02,94.290001,17.240000,7.94
1,1990-01-03,94.419998,18.190001,7.99
2,1990-01-04,92.519997,19.219999,7.98
3,1990-01-05,92.849998,20.110001,7.99
4,1990-01-08,92.050003,20.260000,8.02


In [63]:
# Building macro features and merging into features_df.

# Reindex macro data onto the trading-day calendar from features_df, forward-fill
# small gaps (US holidays vs commodity holidays differ by 1-2 days).
trading_days = features_df["date"].drop_duplicates().sort_values()
macro_aligned = (
    macro_df.set_index("index")
    .reindex(trading_days)
    .ffill(limit=2)
)

# DXY: log returns over 5d and 20d, plus its correlation with the energy basket.
dxy_ret = np.log(macro_aligned["dxy"]).diff()
dxy_ret_5d  = dxy_ret.rolling(5).sum()
dxy_ret_20d = dxy_ret.rolling(20).sum()

# Energy basket return (mean of CL, HO, RB, NG log returns) for the DXY correlation
# This way, the dxy_corr_60d feature is meaningful for all four energy instruments.
cl_ret = np.log(cl_close).diff()
ho_ret = np.log(ho_close).diff()
rb_ret = np.log(rb_close).diff()
ng_ret = np.log(ng_close).diff()
energy_basket_ret = pd.concat([cl_ret, ho_ret, rb_ret, ng_ret], axis=1).mean(axis=1)
energy_basket_ret = energy_basket_ret.reindex(trading_days)

dxy_corr_60d = energy_basket_ret.rolling(60).corr(dxy_ret)

# VIX: level and 5-day change
vix_level     = macro_aligned["vix"]
vix_change_5d = macro_aligned["vix"].diff(5)

# US 10y: 5-day change in yield
us10y_change_5d = macro_aligned["us10y"].diff(5)

# Assemble and merge
macro_metrics_df = pd.DataFrame({
    "dxy_ret_5d":      dxy_ret_5d,
    "dxy_ret_20d":     dxy_ret_20d,
    "dxy_corr_60d":    dxy_corr_60d,
    "vix_level":       vix_level,
    "vix_change_5d":   vix_change_5d,
    "us10y_change_5d": us10y_change_5d,
}).reset_index().rename(columns={"index": "date"})

features_df = features_df.merge(macro_metrics_df, on="date", how="left")

features_df[[
    "date", "instrument",
    "dxy_ret_5d", "dxy_ret_20d", "dxy_corr_60d",
    "vix_level", "vix_change_5d", "us10y_change_5d",
]].head()

,date,instrument,dxy_ret_5d,dxy_ret_20d,dxy_corr_60d,vix_level,vix_change_5d,us10y_change_5d
0,1990-01-02,cl1s,NaN,NaN,NaN,17.240000,NaN,NaN
1,1990-01-03,cl1s,NaN,NaN,NaN,18.190001,NaN,NaN
2,1990-01-04,cl1s,NaN,NaN,NaN,19.219999,NaN,NaN
3,1990-01-05,cl1s,NaN,NaN,NaN,20.110001,NaN,NaN
4,1990-01-08,cl1s,NaN,NaN,NaN,20.260000,NaN,NaN


## 2.15 Feature Set Summary

Before moving to latent regime features, we summarize the feature set created so far and inspect the amount of missing data generated by rolling-window calculations.

In [64]:
# ============================================================
# Feature Set Summary
# ============================================================

feature_columns = [
    col for col in features_df.columns
    if col not in [
        "date", "instrument", "open", "high", "low", "close",
        "volume", "open_interest"
    ]
]

print("Number of engineered features:", len(feature_columns))
print("\nFeature columns:")
for col in feature_columns:
    print("-", col)

missing_summary = (
    features_df[feature_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_ratio")
)

display(missing_summary.head(40))

Number of engineered features: 123

Feature columns:
- log_return
- momentum_5d
- momentum_10d
- momentum_20d
- momentum_60d
- mean_return_5d
- mean_return_20d
- mean_return_60d
- ret_20d_zscore
- realized_vol_5d
- realized_vol_20d
- realized_vol_60d
- ewma_vol_10d
- ewma_vol_20d
- vol_ratio_20_60
- garman_klass_vol
- yang_zhang_vol
- skew_20d
- skew_60d
- kurt_20d
- downside_vol_20d
- price_range_position_60d
- range_position_5d_chg
- return_vol_correl_60d
- vol_parkinson_20d
- vol_rogers_satchell_20d
- log_volume_change
- volume_mean_5d
- volume_std_5d
- volume_mean_20d
- volume_std_20d
- volume_zscore_20d
- log_oi_change
- oi_momentum_5d
- oi_momentum_20d
- amihud_20d
- roll_spread_20d
- dollar_volume_log
- kyle_lambda_20d
- hl_spread
- oc_spread
- close_position
- rsi_14d
- price_zscore_20d
- price_zscore_60d
- macd
- macd_signal
- macd_hist
- atx_14
- distance_from_200d_ma
- willr_14
- stoch_k
- stoch_d
- obv
- mfi_14
- atr_14
- bb_width
- bb_position
- vol_oi_ratio
- autocorr_ret

,missing_ratio
cross_asset_momentum_concordance,0.962072
multiasset_vol_zscore_252d,0.575796
asset_equity_corr_60d,0.563194
relative_energy_vol,0.367940
equity_vol_zscore_252d,0.290213
equity_vol_avg_20d,0.263598
equity_avg_vol_20d,0.237597
risk_on_score,0.235788
equity_avg_ret_5d,0.235758
close_fracdiff,0.178696


## 2.16 Dynamic Cross-Sectional Features: Basket Correlation, Lead-Lag, and Beta

The features computed earlier in this section (`momentum_rank_*`, `relative_vol_*`, `sector_*_dispersion_*`, `relative_momentum_*`) capture **static** cross-sectional positioning: at each date, they describe where an instrument sits *relative to its peers in that moment*.

We now add three **dynamic** cross-sectional features that capture how an instrument moves *with* (or *ahead of*) the rest of the sector over time:

- **`corr_basket_60d`** — 60-day rolling correlation between the instrument's daily log returns and an equal-weighted basket of its peer returns. Captures *co-movement strength*: a high value means the instrument is strongly tied to sector-wide moves, a low value means it is currently decoupled from its peers.

- **`leadlag_anchor`** — 60-day correlation between the instrument's daily log returns and the **one-day-lagged** return of the anchor (CL). Captures whether CL's movement *today* predicts the instrument's movement *tomorrow*. A high positive value indicates the anchor leads, which is empirically a feature of energy markets where crude oil sets the tone for refined products.

- **`beta_basket_60d`** — 60-day rolling OLS beta of the instrument's returns on the basket return. Distinct from `corr_basket_60d` because beta also incorporates the relative volatility of the instrument vs. the basket: an instrument can be highly correlated with the basket (corr ≈ 1) but have beta ≠ 1 if its volatility differs.

Together, these complement the static cross-sectional features by adding temporal structure. The motivation is that the reliability of a primary signal may depend not only on the instrument's *current ranking* within the sector but also on whether it is *currently behaving like* the sector (high correlation) or *diverging* from it (low correlation, unusual beta).

In [65]:
# ============================================================
# Dynamic Cross-Sectional Features: Basket Correlation, Lead-Lag, and Beta
# ============================================================

ANCHOR = "cl1s"

# ------------------------------------------------------------
# Build the peer-basket return (equal-weighted mean of all OTHER instruments)
# at each date, using the leave-one-out trick:
#  peer_basket = (sum_of_all - own) / (n_instruments - 1)
# ------------------------------------------------------------

n_instruments_per_date = (
    features_df
    .groupby("date")["log_return"]
    .transform("count")
)

sum_returns_per_date = (
    features_df
    .groupby("date")["log_return"]
    .transform("sum")
)

features_df["peer_basket_return"] = (
    (sum_returns_per_date - features_df["log_return"])
    / (n_instruments_per_date - 1)
)

# ------------------------------------------------------------
# Feature 1: 60-day rolling correlation with peer basket
# ------------------------------------------------------------

features_df["corr_basket_60d"] = (
    features_df
    .groupby("instrument", group_keys=False)
    .apply(
        lambda g: g["log_return"].rolling(60).corr(g["peer_basket_return"])
    )
)

# ------------------------------------------------------------
# Feature 2: 60-day rolling correlation with lagged anchor (CL)
# ------------------------------------------------------------

# Extract the anchor's daily return series, shifted by one day
anchor_lagged_return = (
    features_df.loc[features_df["instrument"] == ANCHOR]
    .set_index("date")["log_return"]
    .shift(1)
)

# Map the lagged anchor onto every row by date
features_df["anchor_lagged_return"] = (
    features_df["date"].map(anchor_lagged_return)
)

features_df["leadlag_anchor"] = (
    features_df
    .groupby("instrument", group_keys=False)
    .apply(
        lambda g: (
            g["log_return"].rolling(60).corr(g["anchor_lagged_return"])
            if g.name != ANCHOR
            else pd.Series(np.nan, index=g.index)
        )
    )
)

# ------------------------------------------------------------
# Feature 3: 60-day rolling beta on the peer basket
#     beta = Cov(r, b) / Var(b)
# ------------------------------------------------------------

def _rolling_beta(g, window=60):
    cov_rb = (
        (g["log_return"] * g["peer_basket_return"]).rolling(window).mean()
        - g["log_return"].rolling(window).mean()
          * g["peer_basket_return"].rolling(window).mean()
    )
    var_b = g["peer_basket_return"].rolling(window).var()
    return cov_rb / var_b.replace(0, np.nan)

features_df["beta_basket_60d"] = (
    features_df
    .groupby("instrument", group_keys=False)
    .apply(_rolling_beta)
)

# ------------------------------------------------------------
# Clean up temporary columns
# ------------------------------------------------------------

features_df = features_df.drop(
    columns=["peer_basket_return", "anchor_lagged_return"]
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    features_df[
        [
            "date",
            "instrument",
            "corr_basket_60d",
            "leadlag_anchor",
            "beta_basket_60d",
        ]
    ]
    .dropna()
    .head()
)

/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/1350619385.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/1350619385.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/hp/cqvvx0r926vgtbg5wqzgstk40000gn/T/ipykernel_39467/1350619385.py:87: FutureWarning: DataFrameGroupBy.apply operated on the grouping 

,date,instrument,corr_basket_60d,leadlag_anchor,beta_basket_60d
8232,1990-03-29,ho1s,0.791244,-0.093951,1.054563
8233,1990-03-30,ho1s,0.799236,-0.012302,1.019448
8234,1990-04-02,ho1s,0.799514,-0.031358,0.961691
8235,1990-04-03,ho1s,0.777760,-0.059761,0.997529
8236,1990-04-04,ho1s,0.790632,-0.015406,1.041222


## 2.17 HMM Regime Features

We now add latent regime features using a Hidden Markov Model (HMM).

Unlike technical indicators, HMM features are not directly computed from prices. Instead, the HMM learns hidden market states from observable variables such as returns, volatility, and momentum.

To avoid look-ahead bias, the HMM will be fitted only on the training period for each instrument. The fitted model will then be used to infer regime probabilities for both the training and test periods.

### 2.16.1 Define HMM Inputs

We use a small set of interpretable variables to train the HMM:

- daily log return,
- 20-day realized volatility,
- 20-day momentum.

These variables allow the HMM to distinguish between calm, volatile, trending, and stressed market regimes.

In [66]:
# ============================================================
# Define HMM input features
# ============================================================

HMM_FEATURES = [
    "log_return",
    "realized_vol_20d",
    "momentum_20d"
]

N_HMM_STATES = 3

print("HMM input features:", HMM_FEATURES)
print("Number of HMM regimes:", N_HMM_STATES)

HMM input features: ['log_return', 'realized_vol_20d', 'momentum_20d']
Number of HMM regimes: 3


### 2.16.2 Define Train/Test Split for HMM

Before fitting the HMM, we define a chronological train/test split using the primary signal dates.

For each instrument, the first 80% of signal dates are assigned to the training period, while the last 20% are reserved as the final out-of-sample test period.

The HMM is then fitted only using market data up to the last training signal date for each instrument.

In [67]:
# ============================================================
# Define chronological 80/20 split from primary signals
# ============================================================

TEST_SIZE = 0.20

signal_split_info = {}

for instrument in ENERGY_INSTRUMENTS:
    
    signal_dates = (
        signals_long
        .loc[signals_long["instrument"] == instrument, "date"]
        .sort_values()
        .drop_duplicates()
        .reset_index(drop=True)
    )
    
    split_idx = int(len(signal_dates) * (1 - TEST_SIZE))
    
    train_dates = signal_dates.iloc[:split_idx]
    test_dates = signal_dates.iloc[split_idx:]
    
    signal_split_info[instrument] = {
        "n_dates": len(signal_dates),
        "n_train_dates": len(train_dates),
        "n_test_dates": len(test_dates),
        "train_start": train_dates.min(),
        "train_end": train_dates.max(),
        "test_start": test_dates.min(),
        "test_end": test_dates.max()
    }

split_summary = pd.DataFrame(signal_split_info).T
display(split_summary)

,n_dates,n_train_dates,n_test_dates,train_start,train_end,test_start,test_end
cl1s,645,516,129,2020-01-03 00:00:00,2021-12-30 00:00:00,2021-12-31 00:00:00,2022-06-30 00:00:00
ho1s,645,516,129,2020-01-03 00:00:00,2021-12-30 00:00:00,2021-12-31 00:00:00,2022-06-30 00:00:00
rb1s,645,516,129,2020-01-03 00:00:00,2021-12-30 00:00:00,2021-12-31 00:00:00,2022-06-30 00:00:00
ng1s,645,516,129,2020-01-03 00:00:00,2021-12-30 00:00:00,2021-12-31 00:00:00,2022-06-30 00:00:00


In [68]:
global_train_end_date = min(
    info["train_end"] for info in signal_split_info.values()
)

print("Global train_end_date:", global_train_end_date)

Global train_end_date: 2021-12-30 00:00:00


### 2.16.3 Train One HMM per Instrument

We fit one Gaussian HMM per energy instrument.

Each HMM is trained only on the training period and then used to infer latent regimes over the full sample.

The inferred regimes will later be used as additional features for the metamodel.

In [69]:
# ============================================================
# Initialize HMM outputs
# ============================================================
features_df["hmm_regime"] = np.nan

for state in range(N_HMM_STATES):
    features_df[f"hmm_prob_state_{state}"] = np.nan

In [70]:
# ============================================================
# Train HMMs and infer regimes
# ============================================================

hmm_models = {}
hmm_scalers = {}

for instrument in ENERGY_INSTRUMENTS:
    
    print(f"\nTraining HMM for {instrument}...")
    
    # --------------------------------------------------------
    # Instrument subset
    # --------------------------------------------------------
    
    instrument_df = (
        features_df
        .loc[features_df["instrument"] == instrument]
        .sort_values("date")
        .copy()
    )
    
    # --------------------------------------------------------
    # Keep only rows with valid HMM features
    # --------------------------------------------------------
    
    instrument_df = instrument_df.dropna(subset=HMM_FEATURES)
    
    # --------------------------------------------------------
    # Train/test cutoff
    # --------------------------------------------------------
    
    train_end_date = global_train_end_date
    
    train_mask = instrument_df["date"] <= train_end_date
    
    train_df = instrument_df.loc[train_mask]
    
    # --------------------------------------------------------
    # Extract matrices
    # --------------------------------------------------------
    
    X_train = train_df[HMM_FEATURES].values
    X_full = instrument_df[HMM_FEATURES].values
    
    # --------------------------------------------------------
    # Standardize using TRAIN ONLY
    # --------------------------------------------------------
    
    scaler = StandardScaler()
    
    scaler.fit(X_train)
    
    X_train_scaled = scaler.transform(X_train)
    X_full_scaled = scaler.transform(X_full)
    
    hmm_scalers[instrument] = scaler
    
    # --------------------------------------------------------
    # Fit Gaussian HMM
    # --------------------------------------------------------
    
    hmm = GaussianHMM(
        n_components=N_HMM_STATES,
        covariance_type="full",
        n_iter=300,
        random_state=42
    )
    
    hmm.fit(X_train_scaled)
    
    hmm_models[instrument] = hmm
    
    # --------------------------------------------------------
    # Infer regimes on full sample
    # --------------------------------------------------------
    
    hidden_states = hmm.predict(X_full_scaled)
    
    state_probs = hmm.predict_proba(X_full_scaled)
    
    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------
    
    features_df.loc[instrument_df.index, "hmm_regime"] = hidden_states
    
    for state in range(N_HMM_STATES):
        
        features_df.loc[
            instrument_df.index,
            f"hmm_prob_state_{state}"
        ] = state_probs[:, state]
    
    print("Done.")


Training HMM for cl1s...
Done.

Training HMM for ho1s...
Done.

Training HMM for rb1s...
Done.

Training HMM for ng1s...
Done.


### 2.16.4 Inspect HMM Regime Features

We now inspect the HMM regime features to verify that:

- each observation is assigned a regime,
- regime probabilities sum to one,
- regimes are used across instruments,
- and the inferred states can later be interpreted economically.

In [71]:
# ============================================================
# Inspect HMM feature columns
# ============================================================

hmm_cols = [
    "hmm_regime",
    "hmm_prob_state_0",
    "hmm_prob_state_1",
    "hmm_prob_state_2"
]

display(
    features_df[
        ["date", "instrument"] + HMM_FEATURES + hmm_cols
    ]
    .dropna(subset=hmm_cols)
    .head(20)
)

,date,instrument,log_return,realized_vol_20d,momentum_20d,hmm_regime,hmm_prob_state_0,hmm_prob_state_1,hmm_prob_state_2
20,1990-01-30,cl1s,-0.015025,0.025137,0.028225,0.0,1.000000,2.932954e-221,0.000000e+00
21,1990-01-31,cl1s,0.009748,0.024049,0.004042,0.0,0.999991,5.401013e-07,8.793541e-06
22,1990-02-01,cl1s,0.000881,0.023891,0.016391,0.0,0.999991,9.058377e-07,8.036528e-06
23,1990-02-02,cl1s,0.013998,0.023790,0.044586,0.0,0.999977,1.428546e-05,9.056394e-06
24,1990-02-05,cl1s,-0.027749,0.019215,0.082185,0.0,0.993058,6.926827e-03,1.506096e-05
25,1990-02-06,cl1s,0.005345,0.018825,0.066930,0.0,0.844526,1.554706e-01,3.667368e-06
26,1990-02-07,cl1s,-0.008477,0.017233,0.021535,1.0,0.519524,4.804732e-01,3.127406e-06
27,1990-02-08,cl1s,-0.010358,0.017266,0.000751,1.0,0.334728,6.652697e-01,1.876946e-06
28,1990-02-09,cl1s,-0.015971,0.017634,-0.014787,1.0,0.209549,7.904511e-01,3.492557e-07
29,1990-02-12,cl1s,0.013251,0.016053,0.032321,1.0,0.039263,9.607367e-01,7.845380e-08


In [72]:
# ============================================================
# Check HMM probability sums
# ============================================================

features_df["hmm_prob_sum"] = (
    features_df["hmm_prob_state_0"]
    + features_df["hmm_prob_state_1"]
    + features_df["hmm_prob_state_2"]
)

display(
    features_df[
        ["date", "instrument", "hmm_prob_sum"]
    ]
    .dropna()
    .head(20)
)

print(
    "Min probability sum:",
    features_df["hmm_prob_sum"].min()
)

print(
    "Max probability sum:",
    features_df["hmm_prob_sum"].max()
)

,date,instrument,hmm_prob_sum
20,1990-01-30,cl1s,1.0
21,1990-01-31,cl1s,1.0
22,1990-02-01,cl1s,1.0
23,1990-02-02,cl1s,1.0
24,1990-02-05,cl1s,1.0
25,1990-02-06,cl1s,1.0
26,1990-02-07,cl1s,1.0
27,1990-02-08,cl1s,1.0
28,1990-02-09,cl1s,1.0
29,1990-02-12,cl1s,1.0


Min probability sum: 0.9999999999981811
Max probability sum: 1.0000000000018188


### 2.16.5 Macro HMM

The per-instrument HMM in 2.8.3 captures regimes specific to each energy contract. We now fit a **second** HMM on a cross-asset macro state vector — this instrument's return, copper's return, the equity-average return, and the equity volatility level.

The macro HMM is the **same for every energy instrument** because its inputs describe the global macro state, not the instrument's idiosyncratic behaviour. We still fit it once per instrument (because the first input is instrument-specific), but the regimes it identifies will be macro regimes — risk-on / risk-off / stressed — rather than oil-supply or NG-storage regimes.

This complements 2.8.3: features from both HMMs together let the metamodel condition on **both** the instrument-specific and the macro-wide regime. As before, the model is fitted only on the training window to prevent look-ahead.

In [73]:
# ============================================================
# Define Macro HMM inputs
# ============================================================

# The macro HMM uses four cross-asset features:
#   - this instrument's daily log return (idiosyncratic anchor)
#   - copper's daily log return (industrial demand proxy)
#   - equity-average daily log return (risk-on/off proxy)
#   - equity-average 20-day volatility (macro vol regime)
#
# The last three are already in features_df from section 2.8 (Cross-Asset Regime),
# but we recompute the equity-average daily return here since features_df only
# stores the 5-day sum.

# Daily log returns for copper and the three equity index futures
hg_ret_daily = np.log(hg_close).diff().rename("hg_ret_daily")
es_ret_daily = np.log(es_close).diff()
nq_ret_daily = np.log(nq_close).diff()
fesx_ret_daily = np.log(fesx_close).diff()
equity_ret_daily = pd.concat(
    [es_ret_daily, nq_ret_daily, fesx_ret_daily], axis=1
).mean(axis=1).rename("equity_ret_daily")

# equity_avg_vol_20d already exists in features_df (one value per date)
# Build a clean date-indexed macro frame to merge in
macro_hmm_inputs = pd.concat(
    [hg_ret_daily, equity_ret_daily],
    axis=1,
).reset_index()

features_df = features_df.merge(macro_hmm_inputs, on="date", how="left")

# Define the macro HMM input feature list
HMM_MACRO_FEATURES = [
    "log_return",           # this instrument's return (already in features_df)
    "hg_ret_daily",         # copper's return
    "equity_ret_daily",     # equity-average return
    "equity_avg_vol_20d",   # equity vol level (already in features_df from 2.8)
]

N_HMM_MACRO_STATES = 2

print("Macro HMM input features:", HMM_MACRO_FEATURES)
print("Number of macro HMM regimes:", N_HMM_MACRO_STATES)

Macro HMM input features: ['log_return', 'hg_ret_daily', 'equity_ret_daily', 'equity_avg_vol_20d']
Number of macro HMM regimes: 2


In [74]:
# ============================================================
# Initialize Macro HMM outputs
# ============================================================

for state in range(N_HMM_MACRO_STATES):
    features_df[f"hmm_macro_prob_state_{state}"] = np.nan

features_df["hmm_macro_regime"] = np.nan

In [75]:
# ============================================================
# Train Macro HMMs and infer regimes
# ============================================================

hmm_macro_models = {}
hmm_macro_scalers = {}

for instrument in ENERGY_INSTRUMENTS:
    
    print(f"\nTraining Macro HMM for {instrument}...")
    
    # --------------------------------------------------------
    # Instrument subset
    # --------------------------------------------------------
    
    instrument_df = (
        features_df
        .loc[features_df["instrument"] == instrument]
        .sort_values("date")
        .copy()
    )
    
    # --------------------------------------------------------
    # Keep only rows with valid macro HMM features
    # --------------------------------------------------------
    
    instrument_df = instrument_df.dropna(subset=HMM_MACRO_FEATURES)
    
    if len(instrument_df) < 100:
        print(f"  Skipping {instrument}: only {len(instrument_df)} valid rows.")
        continue
    
    # --------------------------------------------------------
    # Train/test cutoff (reuse the split from section 2.8.2)
    # --------------------------------------------------------
    
    train_end_date = global_train_end_date
    
    train_mask = instrument_df["date"] <= train_end_date
    
    train_df = instrument_df.loc[train_mask]
    
    # --------------------------------------------------------
    # Extract matrices
    # --------------------------------------------------------
    
    X_train = train_df[HMM_MACRO_FEATURES].values
    X_full = instrument_df[HMM_MACRO_FEATURES].values
    
    # --------------------------------------------------------
    # Standardize using TRAIN ONLY
    # --------------------------------------------------------
    
    scaler = StandardScaler()
    
    scaler.fit(X_train)
    
    X_train_scaled = scaler.transform(X_train)
    X_full_scaled = scaler.transform(X_full)
    
    hmm_macro_scalers[instrument] = scaler
    
    # --------------------------------------------------------
    # Fit Gaussian HMM
    # --------------------------------------------------------
    
    hmm_macro = GaussianHMM(
        n_components=N_HMM_MACRO_STATES,
        covariance_type="full",
        n_iter=300,
        random_state=42
    )
    
    hmm_macro.fit(X_train_scaled)
    
    hmm_macro_models[instrument] = hmm_macro
    
    # --------------------------------------------------------
    # Infer regimes on full sample
    # --------------------------------------------------------
    
    hidden_states = hmm_macro.predict(X_full_scaled)
    
    state_probs = hmm_macro.predict_proba(X_full_scaled)
    
    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------
    
    features_df.loc[instrument_df.index, "hmm_macro_regime"] = hidden_states
    
    for state in range(N_HMM_MACRO_STATES):
        
        features_df.loc[
            instrument_df.index,
            f"hmm_macro_prob_state_{state}"
        ] = state_probs[:, state]
    
    print("Done.")


Training Macro HMM for cl1s...
Done.

Training Macro HMM for ho1s...
Done.

Training Macro HMM for rb1s...
Done.

Training Macro HMM for ng1s...
Done.


In [76]:
# ============================================================
# Inspect Macro HMM feature columns
# ============================================================

hmm_macro_cols = (
    ["hmm_macro_regime"]
    + [f"hmm_macro_prob_state_{k}" for k in range(N_HMM_MACRO_STATES)]
)

display(
    features_df[
        ["date", "instrument"] + HMM_MACRO_FEATURES + hmm_macro_cols
    ]
    .dropna(subset=hmm_macro_cols)
    .head(20)
)

# Probability sum check
features_df["hmm_macro_prob_sum"] = sum(
    features_df[f"hmm_macro_prob_state_{k}"] for k in range(N_HMM_MACRO_STATES)
)

print(
    "Min macro probability sum:",
    features_df["hmm_macro_prob_sum"].min()
)
print(
    "Max macro probability sum:",
    features_df["hmm_macro_prob_sum"].max()
)

,date,instrument,log_return,hg_ret_daily,equity_ret_daily,equity_avg_vol_20d,hmm_macro_regime,hmm_macro_prob_state_0,hmm_macro_prob_state_1
1954,1997-10-07,cl1s,0.001367,0.005816,0.008118,0.161827,0.0,1.000000e+00,5.837555e-112
1955,1997-10-08,cl1s,0.009968,-0.006346,-0.007354,0.143285,0.0,9.999915e-01,8.503017e-06
1956,1997-10-09,cl1s,-0.002709,0.008452,-0.004336,0.140721,0.0,9.999942e-01,5.808493e-06
1957,1997-10-10,cl1s,-0.000905,-0.008452,-0.001279,0.131660,0.0,9.999962e-01,3.803556e-06
1958,1997-10-13,cl1s,-0.035932,-0.001062,-0.000512,0.130938,0.0,9.999861e-01,1.389760e-05
1959,1997-10-14,cl1s,-0.029512,0.017895,0.001280,0.096742,0.0,9.999824e-01,1.758210e-05
1960,1997-10-15,cl1s,-0.006300,0.012442,-0.004100,0.098447,0.0,9.999927e-01,7.265664e-06
1961,1997-10-16,cl1s,0.018665,-0.017148,-0.014225,0.112012,0.0,9.999542e-01,4.584008e-05
1962,1997-10-17,cl1s,-0.016735,0.003662,-0.011528,0.118754,0.0,9.999592e-01,4.075839e-05
1963,1997-10-20,cl1s,0.008163,0.012455,0.014129,0.127932,0.0,9.997141e-01,2.858574e-04


Min macro probability sum: 0.9999999999981813
Max macro probability sum: 1.000000000001819


## 2.17 Merge Features with Primary Signals

We now merge the engineered feature set with the primary trading signals.

The resulting dataframe contains one row per `(date, instrument)` where the primary model provides a signal. This will become the base dataset used for triple-barrier labeling and later metamodel training.

At this stage, we keep both active signals (`+1`, `-1`) and inactive signals (`0`). The active signals will be used for labeling trades, while inactive signals ????.

In [77]:
# ============================================================
# Merge engineered features with primary signals
# ============================================================

model_base_df = (
    signals_long
    .merge(
        features_df,
        on=["date", "instrument"],
        how="left"
    )
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Model base dataframe shape:", model_base_df.shape)

display(
    model_base_df[
        [
            "date",
            "instrument",
            "primary_signal",
            "close",
            "log_return",
            "momentum_20d",
            "realized_vol_20d",
            "hmm_regime",
            "hmm_prob_state_0",
            "hmm_prob_state_1",
            "hmm_prob_state_2"
        ]
    ].head(20)
)

Model base dataframe shape: (2580, 146)


,date,instrument,primary_signal,close,log_return,momentum_20d,realized_vol_20d,hmm_regime,hmm_prob_state_0,hmm_prob_state_1,hmm_prob_state_2
0,2020-01-03,cl1s,0,25.553469,0.030108,0.077598,0.009621,1.0,1.261281e-06,0.999999,9.101477e-11
1,2020-01-06,cl1s,0,25.642633,0.003483,0.081081,0.009578,1.0,4.282585e-07,1.000000,8.010337e-10
2,2020-01-07,cl1s,-1,25.411618,-0.009050,0.058939,0.009757,1.0,7.279311e-06,0.999993,4.187047e-08
3,2020-01-08,cl1s,0,24.159275,-0.050538,0.011446,0.015425,1.0,4.974155e-03,0.995026,2.024608e-07
4,2020-01-09,cl1s,0,24.139011,-0.000839,0.006887,0.015410,1.0,5.619070e-04,0.999438,3.561078e-09
5,2020-01-10,cl1s,0,23.928261,-0.008769,0.006253,0.015429,1.0,1.413581e-04,0.999859,1.039748e-09
6,2020-01-13,cl1s,0,23.539183,-0.016394,-0.017263,0.015774,1.0,1.729950e-04,0.999827,1.295538e-09
7,2020-01-14,cl1s,0,23.599977,0.002579,-0.029610,0.015360,1.0,1.609793e-04,0.999839,2.924494e-09
8,2020-01-15,cl1s,0,23.429843,-0.007235,-0.039510,0.015379,1.0,5.410658e-04,0.999459,1.712975e-08
9,2020-01-16,cl1s,0,23.709348,0.011859,-0.039716,0.015369,1.0,2.895077e-03,0.997105,1.238839e-07


In [78]:
# ============================================================
# Check merge quality
# ============================================================

print("Missing close after merge:", model_base_df["close"].isna().sum())
print("Missing log_return after merge:", model_base_df["log_return"].isna().sum())
print("Missing HMM regime after merge:", model_base_df["hmm_regime"].isna().sum())

print("\nPrimary signal distribution:")
display(
    model_base_df["primary_signal"]
    .value_counts()
    .sort_index()
)

print("\nPrimary signal distribution by instrument:")
display(
    pd.crosstab(
        model_base_df["instrument"],
        model_base_df["primary_signal"]
    )
)

Missing close after merge: 68
Missing log_return after merge: 68
Missing HMM regime after merge: 68

Primary signal distribution:


primary_signal
-1     431
 0    1343
 1     806
Name: count, dtype: int64


Primary signal distribution by instrument:


primary_signal,-1,0,1
instrument,,,
cl1s,36,223,386
ho1s,10,582,53
ng1s,124,521,0
rb1s,261,17,367


In [79]:
# Rows where signals exist but no matching OHLCV/features were found
missing_feature_rows = model_base_df[model_base_df["close"].isna()].copy()

print("Number of rows with missing features:", len(missing_feature_rows))

display(
    missing_feature_rows[
        ["date", "instrument", "primary_signal"]
    ]
    .sort_values(["instrument", "date"])
    .head(50)
)

Number of rows with missing features: 68


,date,instrument,primary_signal
11,2020-01-20,cl1s,0
31,2020-02-17,cl1s,0
100,2020-05-25,cl1s,0
129,2020-07-03,cl1s,0
175,2020-09-07,cl1s,0
233,2020-11-26,cl1s,0
268,2021-01-18,cl1s,0
288,2021-02-15,cl1s,0
322,2021-04-02,cl1s,0
363,2021-05-31,cl1s,0


In [80]:
# Count missing feature rows by instrument
display(
    missing_feature_rows["instrument"]
    .value_counts()
)

instrument
cl1s    17
ho1s    17
ng1s    17
rb1s    17
Name: count, dtype: int64

In [81]:
# Compare available dates between signals and features for each instrument
for instrument in ENERGY_INSTRUMENTS:
    
    signal_dates = set(
        signals_long.loc[
            signals_long["instrument"] == instrument,
            "date"
        ]
    )
    
    feature_dates = set(
        features_df.loc[
            features_df["instrument"] == instrument,
            "date"
        ]
    )
    
    missing_dates = sorted(signal_dates - feature_dates)
    
    print(f"\n{instrument}")
    print("Missing dates:", len(missing_dates))
    print(missing_dates[:10])


cl1s
Missing dates: 17
[Timestamp('2020-01-20 00:00:00'), Timestamp('2020-02-17 00:00:00'), Timestamp('2020-05-25 00:00:00'), Timestamp('2020-07-03 00:00:00'), Timestamp('2020-09-07 00:00:00'), Timestamp('2020-11-26 00:00:00'), Timestamp('2021-01-18 00:00:00'), Timestamp('2021-02-15 00:00:00'), Timestamp('2021-04-02 00:00:00'), Timestamp('2021-05-31 00:00:00')]

ho1s
Missing dates: 17
[Timestamp('2020-01-20 00:00:00'), Timestamp('2020-02-17 00:00:00'), Timestamp('2020-05-25 00:00:00'), Timestamp('2020-07-03 00:00:00'), Timestamp('2020-09-07 00:00:00'), Timestamp('2020-11-26 00:00:00'), Timestamp('2021-01-18 00:00:00'), Timestamp('2021-02-15 00:00:00'), Timestamp('2021-04-02 00:00:00'), Timestamp('2021-05-31 00:00:00')]

rb1s
Missing dates: 17
[Timestamp('2020-01-20 00:00:00'), Timestamp('2020-02-17 00:00:00'), Timestamp('2020-05-25 00:00:00'), Timestamp('2020-07-03 00:00:00'), Timestamp('2020-09-07 00:00:00'), Timestamp('2020-11-26 00:00:00'), Timestamp('2021-01-18 00:00:00'), Timesta

In [82]:
# ============================================================
# Drop signal rows without matching OHLCV/features
# ============================================================

model_base_df = (
    model_base_df
    .dropna(subset=["close"])
    .copy()
    .reset_index(drop=True)
)

print("Clean model base dataframe shape:", model_base_df.shape)

print("Remaining missing close:", model_base_df["close"].isna().sum())
print("Remaining missing HMM regime:", model_base_df["hmm_regime"].isna().sum())

Clean model base dataframe shape: (2512, 146)
Remaining missing close: 0
Remaining missing HMM regime: 0


## 2.18 Primary Signal Interaction Features

The features built so far describe **market state** — what prices, returns, volatilities, regimes, and cross-sectional positioning look like. None of them describe the **primary signal's own state**.

The metamodel's job is to learn *when the primary signal is reliable*. Reliability is conditional on the signal's own behaviour:

- A signal that just changed direction is different from one that has been steady for weeks.
- A signal that disagrees with the longer-term trend is less reliable than one that agrees.
- A signal that has been firing every day is in a different regime from one that has been mostly flat.

We therefore construct four features describing the signal's own dynamics. These features can only be built **after** the primary signal has been merged with the OHLCV/feature panel, so they live in `model_base_df` rather than `features_df`.

- **`signal_changed`** — binary flag, 1 if today's signal differs from yesterday's.
- **`signal_persistence`** — days since the last signal change.
- **`signal_trend_concord`** — sign(signal) × sign(50-day return). Captures whether the signal aligns with the longer-term trend.
- **`signal_density_20d`** — fraction of the last 20 days where the signal was non-zero. Captures whether the primary model is currently active or quiet.

These are the meta-labelling-specific conditioners that let the classifier learn signal-state-conditional reliability patterns.

In [83]:
# ============================================================
# Primary Signal Interaction Features
# ============================================================

# Feature 1: did the signal change today vs yesterday?
model_base_df["signal_changed"] = (
    model_base_df
    .groupby("instrument")["primary_signal"]
    .transform(lambda x: (x != x.shift(1)).astype(int))
)

# Feature 2: days since the signal last changed (signal persistence)
# Trick: cumulative count of changes defines a "run id", then count within each run
def _persistence(s):
    changes = (s != s.shift(1)).astype(int)
    run_id = changes.cumsum()
    return run_id.groupby(run_id).cumcount()

model_base_df["signal_persistence"] = (
    model_base_df
    .groupby("instrument")["primary_signal"]
    .transform(_persistence)
)

# Feature 3: concordance of signal with 50-day return direction
# Need 50-day log return; build it here from close (already available)
model_base_df["log_return_50d"] = (
    model_base_df
    .groupby("instrument")["close"]
    .transform(lambda x: np.log(x / x.shift(50)))
)

model_base_df["signal_trend_concord"] = (
    np.sign(model_base_df["primary_signal"])
    * np.sign(model_base_df["log_return_50d"])
)

# Drop the helper column
model_base_df = model_base_df.drop(columns=["log_return_50d"])

# Feature 4: fraction of last 20 days where signal was non-zero
model_base_df["signal_density_20d"] = (
    model_base_df
    .groupby("instrument")["primary_signal"]
    .transform(lambda x: (x != 0).rolling(20).mean())
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    model_base_df[
        [
            "date",
            "instrument",
            "primary_signal",
            "signal_changed",
            "signal_persistence",
            "signal_trend_concord",
            "signal_density_20d",
        ]
    ]
    .dropna()
    .head(10)
)

# Quick sanity check: distribution of signal_persistence by instrument
print("\nSignal persistence stats by instrument:")
display(
    model_base_df
    .groupby("instrument")["signal_persistence"]
    .describe()
    .round(1)
)

,date,instrument,primary_signal,signal_changed,signal_persistence,signal_trend_concord,signal_density_20d
50,2020-03-17,cl1s,0,0,11,-0.0,0.40
51,2020-03-18,cl1s,1,1,0,-1.0,0.40
52,2020-03-19,cl1s,0,1,0,-0.0,0.35
53,2020-03-20,cl1s,1,1,0,-1.0,0.35
54,2020-03-23,cl1s,1,0,1,-1.0,0.35
55,2020-03-24,cl1s,1,0,2,-1.0,0.35
56,2020-03-25,cl1s,0,1,0,-0.0,0.30
57,2020-03-26,cl1s,1,1,0,-1.0,0.30
58,2020-03-27,cl1s,1,0,1,-1.0,0.30
59,2020-03-30,cl1s,1,0,2,-1.0,0.35



Signal persistence stats by instrument:


,count,mean,std,min,25%,50%,75%,max
instrument,,,,,,,,
cl1s,628.0,13.4,15.7,0.0,2.0,7.0,19.0,70.0
ho1s,628.0,28.5,30.1,0.0,4.0,16.5,47.0,112.0
ng1s,628.0,35.9,39.0,0.0,6.0,22.0,53.0,159.0
rb1s,628.0,21.4,29.3,0.0,3.0,9.0,27.0,128.0


## Summary Before Phase 3: Dataset Construction and Feature Engineering

At this stage, we have completed the full feature engineering pipeline and built the base dataset for the metamodel.

The workflow implemented so far is summarized below:

```text
Raw OHLCV data
        ↓
Feature Engineering
    - Return and momentum features
    - Volatility features (realized, EWMA, range-based)
    - Higher moments and range position
    - Volume and open interest features
    - Microstructure and liquidity features
    - Technical indicators
    - Time-series dependence features
    - Spectral and fractal features
    - Cross-sectional features
    - Seasonality features
    - Inter-energy spreads
    - Cross-asset regime, correlation, momentum and volatility
    - Metals ratios
    - Copper leading indicators
    - Macro context
        ↓
HMM regime features (per-instrument + macro)
        ↓
Merge with primary trading signals
        ↓
Primary-signal interaction features
        ↓
Base metamodel dataset
```

### Features Created

The engineered features currently include several categories:

**Return and momentum features**
- Momentum over multiple horizons (`momentum_{window}d`)
- Rolling mean returns (`mean_return_{window}d`)
- 20-day return z-score (`ret_20d_zscore`)

**Volatility features**
- Rolling realized volatility (`realized_vol_{window}d`)
- EWMA volatility (`ewma_vol_{span}d`)
- Volatility ratio (`vol_ratio_20_60`)
- Range-based volatility: Garman-Klass (`garman_klass_vol`), Yang-Zhang (`yang_zhang_vol`)
- Downside volatility (`downside_vol_20d`)
- Volatility change (`vol_change_20d`)
- Range-based volatility: Garman-Klass (`garman_klass_vol`), Yang-Zhang (`yang_zhang_vol`), Parkinson (`vol_parkinson_20d`), Rogers-Satchell (`vol_rogers_satchell_20d`)

**Higher moments and range position**
- Rolling skewness (`skew_20d`, `skew_60d`)
- Rolling kurtosis (`kurt_20d`)
- Price range position (`price_range_position_60d`, `range_position_5d_chg`)

**Volume and open interest features**
- Log volume change (`log_volume_change`)
- Rolling volume mean and std (`volume_mean_{window}d`, `volume_std_{window}d`)
- Volume z-score (`volume_zscore_20d`)
- Log open interest change (`log_oi_change`)
- Open interest momentum (`oi_momentum_{window}d`)
- Volume-to-open-interest ratio (`vol_oi_ratio`)

**Microstructure and liquidity features**
- Dollar volume (`dollar_volume`, `dollar_volume_log`)
- Amihud illiquidity (`amihud_daily`, `amihud_20d`)
- Roll spread estimate (`roll_spread_20d`, `cov_pricechanges_20d`)
- Kyle's lambda (`kyle_lambda_20d`)
- Single-bar OHLC spreads (`hl_spread`, `oc_spread`, `close_position`)

**Technical indicators**
- RSI (`rsi_14d`)
- Price z-score (`price_zscore_{window}d`)
- ATR (`atr_14`)
- Distance from 200-day moving average (`distance_from_200d_ma`)
- Bollinger Bands width and position (`bb_width`, `bb_position`)
- Williams %R (`willr_14`)
- Stochastic oscillator (`stoch_k`, `stoch_d`)
- On-Balance Volume (`obv`)
- Money Flow Index (`mfi_14`)

**Time-series dependence features**
- Return autocorrelation (`autocorr_return_{window}d`)
- Absolute return autocorrelation (`autocorr_abs_return_{window}d`)
- Trend persistence / positive return ratio (`positive_return_ratio_{window}d`)

**Spectral and fractal features**
- Hurst exponent (`hurst_90d`)
- Detrended fluctuation analysis (`dfa_alpha_90d`)
- Dominant cycle period (`dominant_cycle_period`)
- Spectral entropy (`spectral_entropy`)
- Approximate entropy (`approx_entropy_20d`)
- Fractional differentiation of close (`close_fracdiff`)
- Supremum ADF / explosive-regime statistic (`sadf`)
- Histogram Shannon entropy (`shannon_entropy_hist_20d`)
- Lempel-Ziv complexity (`lz_complexity_20d`)

**Cross-sectional features**
- Momentum rank (`momentum_rank_20d`, `momentum_rank_60d`)
- Relative momentum (`relative_momentum_20d`)
- Relative volatility (`relative_vol_20d`, `relative_vol_60d`)
- Sector dispersion (`sector_momentum_dispersion_20d`, `sector_momentum_dispersion_60d`, `sector_vol_dispersion_20d`)

**Seasonality features**
- Day-of-year cyclical encoding (`day_of_year_sin`, `day_of_year_cos`)
- Heating season (`heating_season`)
- Driving season progress (`driving_season_progress`)
- Hurricane season indicator (`hurricane_season_indicator`)
- Quarter progress (`quarter_progress`)

**Inter-energy spreads**
- 3-2-1 crack spread and its z-score / change (`crack_321_zscore_252d`, `crack_321_change_5d`)
- Oil/gas ratio and its z-score (`oil_gas_ratio_zscore_252d`)

**Cross-asset features**
- Rolling correlations to copper, gold and equities (`asset_copper_corr_60d`, `asset_gold_corr_60d`, `asset_equity_corr_60d`)
- Cross-asset basket correlation and beta (`corr_basket_60d`, `beta_basket_60d`)
- Cross-asset momentum and concordance (`peer_basket_return`, `relative_momentum_20d`)
- Lead-lag features (`anchor_lagged_return`, `leadlag_anchor`)

**Metals ratios**
- Copper/gold ratio with z-score and change (`copper_gold_ratio_zscore_252d`, `copper_gold_ratio_chg_20d`)
- Gold/silver ratio (`gold_silver_ratio`)

**Copper leading indicators**
- Copper 20-day z-score (`copper_zscore_20d`)
- Copper rolling trend t-value (`copper_trend_t_value`)
- Copper momentum / lead features (`copper_mom_20d`, `copper_lead_5d`)

**Latent regime features (HMM)**

Per-instrument HMM:
- `hmm_regime`
- `hmm_prob_state_0`
- `hmm_prob_state_1`
- `hmm_prob_state_2`

Macro HMM:
- `hmm_macro_regime`
- `hmm_macro_prob_state_0`
- `hmm_macro_prob_state_1`
- `hmm_macro_prob_state_2`

The per-instrument HMM was trained independently for each energy instrument using:

- log returns
- realized volatility
- momentum

A separate macro HMM was trained on cross-asset inputs to capture market-wide regimes shared across instruments.

To avoid look-ahead bias:

- the HMMs were fitted only on the training period,
- standardization was fitted only on training data,
- the trained HMMs were then used to infer regimes over the full sample.

**Primary-signal interaction features**

After merging with the primary signals, we add features that describe the signal itself and its interaction with market state:

- `signal_changed`
- `signal_persistence`
- `signal_trend_concord`
- `signal_density_20d`

---

### Base Metamodel Dataset

The engineered features were merged with the primary trading signals to create:

```python
model_base_df
```

This dataset now contains:

- date
- instrument
- primary signal
- market variables
- engineered features
- HMM regime features (per-instrument and macro)
- primary-signal interaction features

Each row corresponds to one `(date, instrument)` pair where a primary signal exists.

---

### Data Cleaning

During the merge process, some rows contained missing values because certain signal dates corresponded to market holidays or non-trading days.

Examples include:

- Martin Luther King Day
- Presidents’ Day
- Thanksgiving
- Good Friday

These observations were removed because no OHLCV data existed for those dates.

---

### Train/Test Protocol

A chronological train/test split has already been defined and is now frozen.

The split was created from the primary signal dataset:

- first 80% → training period
- last 20% → final out-of-sample period

This split will remain unchanged for:

- HMM training
- Triple-barrier labeling
- Hyperparameter tuning
- Model training
- Final evaluation

No future information has been used during feature construction or HMM fitting.

The next step is to construct the target variable using the Triple-Barrier labeling framework.

# Phase 3 — Triple Barrier Labeling: Methodological Decisions and Design Notes

## Objective

The purpose of Phase 3 is to transform the active primary trading signals into supervised learning labels for the meta-model.

The primary model already predicts the trade direction:

- `+1` → long trade
- `-1` → short trade
- `0` → no trade

The meta-model does **not** try to predict market direction.

Instead, it learns:

$$P(\text{Trade is profitable} \mid \text{Market Context})$$

Its role is therefore:

- filter bad trades
- keep good trades
- potentially adjust confidence or position sizing later

---

## 1. Why inactive signals (`0`) create a problem

Our merged dataset currently contains:

```text
(date, instrument)
```

rows for all observations:

```text
date      instrument      primary_signal
-----------------------------------------
t1         CL1S                 +1
t2         CL1S                  0
t3         CL1S                 -1
t4         NG1S                  0
...
```

A `primary_signal = 0` means:

```text
No trade proposed by the primary model
```

Therefore there is:

- no entry price
- no trade
- no trade outcome
- no profit-taking event
- no stop-loss event

Thus we cannot directly assign:

```python
label = 0 or 1
```

because there is no trade to evaluate.

---

#### Exclude inactive signals from meta-labeling

Create:

```python
meta_df = df[df.primary_signal !=0]
```

while keeping the original dataframe intact.

---

Important:

We are NOT deleting information.

We keep:

```python
df_full
```

containing:

```text
+1
-1
0
```

The inactive observations can still contribute to:

- volatility estimates
- rolling statistics
- HMM regime detection
- feature construction
- descriptive analysis

Only:

```python
meta_df
```

will be used for supervised labels.

---

## 2. Sample size issue

Removing inactive signals dramatically reduces observations.

Example:

```text
HO1S:

-1 : 10
0 : 582
+1 : 53
```

Active trades:

```text
63 only
```

Training one independent model per instrument would therefore become unstable.

---

## 3. Global (pooled) meta-model approach

Instead of:

```text
model_CL1S
model_HO1S
model_NG1S
...
```

we train:

```text
ONE global meta-model
```

across all instruments.

---

### Why this helps

Instead of learning only from:

```text
HO1S → 63 observations
```

the model learns from:

```text
HO1S
CL1S
NG1S
RB1S
ES1S
...
```

giving potentially thousands of trade observations.

This allows:

- more stable estimation
- reduced overfitting
- better generalization

---

## 4. How the model distinguishes instruments

Each observation remains:

```text
(date, instrument)
```

Example:

```text
date      instrument   signal   vol20   mom20   label
-------------------------------------------------------
2020      CL1S          +1      0.15     0.2      1
2020      NG1S          -1      0.60    -0.1      0
2020      HO1S          +1      0.35     0.1      1
```

We include:

```text
instrument
asset_class
primary_signal
```

as features.

The model can therefore learn:

```text
"NG1S short trades perform well under high volatility"

or

"HO1S behaves differently"
```

Instrument information will likely be encoded via:

- one-hot encoding
- categorical encoding

---

## 5. Time ordering and dataframe structure

Current dataframe:

```text
CL1S 2020→2022
HO1S 2020→2022
RB1S 2020→2022
...
```

Dates restart for each instrument.

This is NOT problematic for ML.

However:

Triple-barrier calculations MUST be performed separately for each instrument.

Therefore:

```python
df = df.sort_values(
    ["instrument","date"]
)

df.groupby(
    "instrument"
)
```

will be used.

Otherwise:

future windows could accidentally jump between instruments.

---

Rule:

```text
Labeling:
groupby(instrument)

Model training:
pooled across instruments
```

---

## 6. Train/Validation/Test split

We already fixed the original temporal boundaries earlier.

Important:

These boundaries MUST remain unchanged.

Even after removing rows or constructing labels we do NOT recompute splits.

Instead we keep:

```text
Original train end date (global)
Original validation dates
Original test dates
```

and apply:

```python
train = data[
date < train_end
]

validation = ...

test = ...
```

This avoids look-ahead bias.

---

## 7. Triple Barrier parameters

Three parameters must be selected:

---

### Horizontal barriers

Profit-taking: $PT$

Stop-loss: $SL$

---

### Vertical barrier

Maximum holding period: $h$

---

## 8. Selection of PT and SL


---



## 9. Horizon selection

---

## 10. Handling the vertical barrier outcome

Use return at horizon

Compute:

$$r_h = \frac{P_{t+h}}{P_t}-1$$

Adjusted for trade direction:

$$TradeReturn = Signal \times r_h$$

Then:

```python
if TradeReturn>0:
    label=1
else:
    label=0
```

Advantages:

- keeps information
- avoids excessive data loss
- economically meaningful

## 3.1 Constructing the Meta-Labeling Target

We now construct the supervised target used by the meta-model.

The primary model already provides a trade direction:

- `+1`: long trade
- `-1`: short trade
- `0`: no trade

The meta-model should not create new trades. Its purpose is to decide whether an existing active signal should be trusted or filtered out.

Therefore, triple-barrier labels are only computed for active primary signals (`+1` or `-1`). Inactive signals are kept in the full dataset for feature construction and regime analysis, but they are excluded from the meta-labeling target because no trade is proposed by the primary model.

We keep two datasets:

- `model_base_df`: full dataset after feature merge
- `meta_df`: active-signal dataset used for triple-barrier labeling

In [84]:
# ============================================================
# Phase 3.1 — Construct Active-Signal Meta-Labeling Dataset
# ============================================================

# Keep a full copy for reference, feature analysis, and possible regime work
full_model_df = model_base_df.copy()

# Meta-labeling is only meaningful when the primary model proposes a trade
meta_df = (
    model_base_df
    .loc[model_base_df["primary_signal"] != 0]
    .copy()
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Full model dataframe shape:", full_model_df.shape)
print("Meta-labeling dataframe shape:", meta_df.shape)

print("\nPrimary signal distribution in full dataset:")
display(
    full_model_df["primary_signal"]
    .value_counts()
    .sort_index()
)

print("\nPrimary signal distribution in meta-labeling dataset:")
display(
    meta_df["primary_signal"]
    .value_counts()
    .sort_index()
)

print("\nActive signal distribution by instrument:")
display(
    pd.crosstab(
        meta_df["instrument"],
        meta_df["primary_signal"]
    )
)

Full model dataframe shape: (2512, 150)
Meta-labeling dataframe shape: (1237, 150)

Primary signal distribution in full dataset:


primary_signal
-1     431
 0    1275
 1     806
Name: count, dtype: int64


Primary signal distribution in meta-labeling dataset:


primary_signal
-1    431
 1    806
Name: count, dtype: int64


Active signal distribution by instrument:


primary_signal,-1,1
instrument,,
cl1s,36,386
ho1s,10,53
ng1s,124,0
rb1s,261,367


## 3.2 Temporal Structure Check

Before computing triple-barrier labels, we verify that the active-signal dataset is ordered by instrument and date.

This matters because triple-barrier labeling uses future prices. The future path of a trade must always be taken from the same instrument, never from the next instrument in the stacked dataframe.

Therefore, all labeling functions will be applied using:

```python
groupby("instrument")

In [85]:
# ============================================================
# Phase 3.2 — Temporal Structure Check
# ============================================================

for instrument, g in meta_df.groupby("instrument"):
    is_sorted = g["date"].is_monotonic_increasing
    print(f"{instrument}: sorted by date = {is_sorted}, n_active_signals = {len(g)}")

print("\nDate range by instrument:")
display(
    meta_df
    .groupby("instrument")["date"]
    .agg(["min", "max", "count"])
)

cl1s: sorted by date = True, n_active_signals = 422
ho1s: sorted by date = True, n_active_signals = 63
ng1s: sorted by date = True, n_active_signals = 124
rb1s: sorted by date = True, n_active_signals = 628

Date range by instrument:


,min,max,count
instrument,,,
cl1s,2020-01-07,2022-06-30,422
ho1s,2020-01-21,2022-06-02,63
ng1s,2020-05-05,2022-06-29,124
rb1s,2020-01-03,2022-06-30,628


## 3.3 Choice of Volatility Estimator

The triple-barrier method requires a volatility estimate to define the width of the profit-taking and stop-loss barriers.

Several volatility estimators were considered.

### Option 1 — Rolling realized volatility

$$[\sigma_t=Std(r_{t-n+1},...,r_t)]$$

Advantages:

- simple
- widely used
- easy interpretation
- computationally inexpensive

Disadvantages:

- all observations receive equal weight
- does not explicitly capture volatility clustering

Decision for baseline:

✅ Selected initially

We use:

```python
realized_vol_20d
```

This choice provides a compromise between:

- responsiveness
- stability
- interpretability

and is consistent with an intermediate holding horizon.

---

### Option 2 — EWMA volatility

$$[\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_t^2]$$

Advantages:

- larger weight on recent observations
- captures changing volatility environments

---

### Option 3 — GARCH volatility

Advantages:

- captures volatility clustering

Disadvantages:

- parameter estimation required
- increased complexity

---

### Option 4 — Range-based volatility (ATR / Parkinson / Garman-Klass)

Advantages:

- uses OHLC information
- potentially more informative for futures markets

---

For the initial implementation we adopt:

```python
vol_col = "realized_vol_20d"
```

to establish a simple and robust baseline before exploring more sophisticated alternatives.

## 3.4 Initial Triple-Barrier Parameters

We define an initial baseline specification for the triple-barrier method.

The selected values are intentionally simple and symmetric.

Initial choices:

- Holding horizon: `h = 10`
- Profit-taking multiplier: `PT = 1.5`
- Stop-loss multiplier: `SL = 1.5`
- Volatility estimator: `realized_vol_20d`

These choices are not assumed to be optimal and may later be compared against alternative specifications using validation performance.

In [86]:
# ============================================================
# Phase 3.4 — Initial Triple Barrier Parameters
# ============================================================

# Entry price used for trades
price_col = "close"

# Volatility estimate
vol_col = "realized_vol_20d"

# Vertical barrier
HORIZON = 10

# Horizontal barriers
PT_MULT = 1.5
SL_MULT = 1.5


print("Selected baseline parameters")
print("--------------------------------")
print("Price column:", price_col)
print("Volatility:", vol_col)
print("Holding horizon:", HORIZON)
print("PT multiplier:", PT_MULT)
print("SL multiplier:", SL_MULT)

Selected baseline parameters
--------------------------------
Price column: close
Volatility: realized_vol_20d
Holding horizon: 10
PT multiplier: 1.5
SL multiplier: 1.5


### Why asymmetric pt_mult = 2.0, sl_mult = 1.0

The notebook's baseline used symmetric barriers (`PT = SL = 1.5`). After sensitivity testing on the pooled energy dataset, we revised these to asymmetric `pt_mult = 2.0, sl_mult = 1.0`. The choice is methodologically motivated and empirically validated by the grid below.

#### What the symmetric choice actually labels

With symmetric barriers, the triple-barrier rule effectively asks:

> *Does the price move up by 1.5σ before it moves down by 1.5σ?*

For a price series with a half-decent primary signal this is close to a directional 50/50 question. The label tells the metamodel little more than "did the trade move in the signalled direction first?" — economically trivial.

#### What asymmetric 2:1 labels

With `pt_mult = 2.0, sl_mult = 1.0`, the rule asks:

> *Does the trade reach a profit of 2σ before stopping out at -1σ?*

This question carries economic content: it implicitly asks whether the trade achieves a reward-to-risk ratio of at least 2:1. A trader using the metamodel's probability output for position sizing gets a probability already calibrated to a meaningful reward-risk profile, which is directly tradable.

#### How we found 2:1 specifically

We ran a sensitivity grid (next cell) across:

pt_mult ∈ {1.0, 1.5, 2.0, 2.5, 3.0}
sl_mult ∈ {0.5, 1.0, 1.5}
h       ∈ {5, 10, 15}

scoring each combination on three labelling-quality criteria:
- **Class balance**: positive-label rate near 50%
- **Barrier resolution efficiency**: low share of events resolving at the vertical barrier
- **Holding-period efficiency**: median holding close to h/2

We deliberately did NOT score on Sharpe of trade returns: maximising Sharpe conflates labelling quality with primary-signal profitability and surfaces configurations with poor class balance.

#### Our chosen configuration is the grid winner

| Combination | pct_positive | pct_vertical_barrier | median_holding | composite |
|---|---|---|---|---|
| **2.0/1.0/10 (selected)** | **49.0%** | **2.2%** | **4** | **0.936** |
| 1.0/1.0/5 | 51.3% | 2.3% | 2 | 0.933 |
| 1.5/1.0/10 | 50.1% | 0.9% | 3 | 0.923 |

The 2:1 configuration produces nearly perfect class balance (49.0% positive), the lowest practical vertical-barrier share (2.2%), and a median holding period of 4 days — efficient use of the 10-day window. Empirical performance aligns with the methodology argument: asymmetric 2:1 is both economically meaningful and statistically the best-performing configuration.

#### A methodology note on intraday barrier detection

An earlier version of the grid showed quite different rankings — configurations with tight stops (`sl_mult=0.5`) dominated. The change came from upgrading the barrier-detection logic to use intraday High/Low rather than the daily close. With close-only detection, barriers touched intraday but with prices closing back inside the band were systematically missed, and only configurations with very tight stops triggered fast resolutions. The intraday H/L fix follows AFML §3.5 and corrects this bias, producing more honest barrier-resolution diagnostics.

#### Trade-off acknowledged

The asymmetric choice does have one cost: for high-volatility instruments (NG with 5-7% daily vol), barriers at ±2σ get touched quickly (within 1-3 days), so most NG labels resolve fast with high positive rates. This is a feature of the asset, not the parameter — at any pt_mult, NG resolves quickly because of its volatility level. Per-instrument diagnostics in Part 5 will report this transparently.

## 3.5 Triple-Barrier Labeling Function

We now implement the triple-barrier labeling rule.

For each active trade:

- if the primary signal is `+1`, the trade is long;
- if the primary signal is `-1`, the trade is short.

The barriers are computed using the entry price and the volatility estimate at the trade date.

For a long trade:

- hitting the upper barrier first gives `meta_label = 1`;
- hitting the lower barrier first gives `meta_label = 0`.

For a short trade:

- hitting the lower barrier first gives `meta_label = 1`;
- hitting the upper barrier first gives `meta_label = 0`.

If neither horizontal barrier is reached before the vertical barrier, we use the sign-adjusted return at horizon $h$:

$$\text{trade return} = \text{primary signal} \times \left(\frac{P_{t+h}}{P_t}-1\right)$$

and assign:

$$\text{meta label} = 1\{\text{trade return}>0\}$$

In [87]:
# ============================================================
# Phase 3.5 — Triple Barrier Labeling Function
# ============================================================

def apply_triple_barrier_to_instrument(
    instrument_df,
    price_col,
    vol_col,
    horizon=10,
    pt_mult=1.5,
    sl_mult=1.5
):
    """
    Apply triple-barrier labeling to one instrument at a time.

    Parameters
    ----------
    instrument_df : pd.DataFrame
        DataFrame containing only one instrument, sorted by date.
    price_col : str
        Column used as the trade entry price.
    vol_col : str
        Backward-looking volatility column used to scale barriers.
    horizon : int
        Maximum holding period in number of rows/trading days.
    pt_mult : float
        Profit-taking multiplier.
    sl_mult : float
        Stop-loss multiplier.

    Returns
    -------
    pd.DataFrame
        Same dataframe with triple-barrier outputs added.
    """

    df_inst = instrument_df.copy().sort_values("date").reset_index(drop=True)

    meta_labels = []
    event_types = []
    exit_dates = []
    exit_prices = []
    trade_returns = []

    prices = df_inst[price_col].values
    vols = df_inst[vol_col].values
    signals = df_inst["primary_signal"].values
    dates = df_inst["date"].values

    n = len(df_inst)

    for i in range(n):

        entry_price = prices[i]
        vol_t = vols[i]
        signal = signals[i]

        # If volatility or price is missing, we cannot define reliable barriers
        if pd.isna(entry_price) or pd.isna(vol_t) or vol_t <= 0:
            meta_labels.append(np.nan)
            event_types.append("missing_input")
            exit_dates.append(pd.NaT)
            exit_prices.append(np.nan)
            trade_returns.append(np.nan)
            continue

        # If there is not enough future data to reach the vertical barrier,
        # we cannot label the trade cleanly
        end_i = min(i + horizon, n - 1)

        if end_i == i:
            meta_labels.append(np.nan)
            event_types.append("no_future_data")
            exit_dates.append(pd.NaT)
            exit_prices.append(np.nan)
            trade_returns.append(np.nan)
            continue

        upper_barrier = entry_price * (1 + pt_mult * vol_t)
        lower_barrier = entry_price * (1 - sl_mult * vol_t)

        label = None
        event_type = None
        exit_i = None

        # Scan the future path until the vertical barrier
        for j in range(i + 1, end_i + 1):

            price_j = prices[j]

            if signal == 1:
                # Long trade
                if price_j >= upper_barrier:
                    label = 1
                    event_type = "profit_taking"
                    exit_i = j
                    break

                elif price_j <= lower_barrier:
                    label = 0
                    event_type = "stop_loss"
                    exit_i = j
                    break

            elif signal == -1:
                # Short trade
                if price_j <= lower_barrier:
                    label = 1
                    event_type = "profit_taking"
                    exit_i = j
                    break

                elif price_j >= upper_barrier:
                    label = 0
                    event_type = "stop_loss"
                    exit_i = j
                    break

        # If no horizontal barrier is hit, use vertical barrier outcome
        if label is None:
            exit_i = end_i
            exit_price = prices[exit_i]

            realized_trade_return = signal * (exit_price / entry_price - 1)

            label = int(realized_trade_return > 0)
            event_type = "vertical_barrier"

        else:
            exit_price = prices[exit_i]
            realized_trade_return = signal * (exit_price / entry_price - 1)

        meta_labels.append(label)
        event_types.append(event_type)
        exit_dates.append(dates[exit_i])
        exit_prices.append(exit_price)
        trade_returns.append(realized_trade_return)

    df_inst["meta_label"] = meta_labels
    df_inst["tb_event_type"] = event_types
    df_inst["tb_exit_date"] = exit_dates
    df_inst["tb_exit_price"] = exit_prices
    df_inst["tb_trade_return"] = trade_returns

    return df_inst

In [88]:
# ============================================================
# Phase 3.6 — Apply Triple Barrier Labeling
# ============================================================

labeled_groups = []

for instrument, g in meta_df.groupby("instrument"):
    g = g.copy()
    g["instrument"] = instrument  # keep instrument explicitly as a column
    
    labeled_g = apply_triple_barrier_to_instrument(
        g,
        price_col=price_col,
        vol_col=vol_col,
        horizon=HORIZON,
        pt_mult=PT_MULT,
        sl_mult=SL_MULT
    )
    
    labeled_g["instrument"] = instrument
    labeled_groups.append(labeled_g)

meta_labeled_df = (
    pd.concat(labeled_groups, axis=0)
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Final labeled dataset shape:")
print(meta_labeled_df.shape)

print("\nColumns check:")
print("instrument" in meta_labeled_df.columns)

print("\nMeta-label distribution:")
display(
    meta_labeled_df["meta_label"]
    .value_counts(dropna=False)
)

print("\nBarrier event distribution:")
display(
    meta_labeled_df["tb_event_type"]
    .value_counts(dropna=False)
)

Final labeled dataset shape:
(1237, 155)

Columns check:
True

Meta-label distribution:


meta_label
1.0    690
0.0    543
NaN      4
Name: count, dtype: int64


Barrier event distribution:


tb_event_type
profit_taking       625
stop_loss           490
vertical_barrier    118
no_future_data        4
Name: count, dtype: int64

## 3.6 Apply Triple-Barrier Labeling

The labeling function is now applied separately to each instrument.

This ensures that:

- future paths are computed only within the same instrument;
- dates remain temporally consistent;
- barrier calculations use instrument-specific volatility dynamics.

The resulting dataset becomes the final supervised dataset used later for meta-model training.

In [89]:
# ============================================================
# Phase 3.6 — Apply Triple Barrier Labeling
# ============================================================

labeled_groups = []

for instrument, g in meta_df.groupby("instrument"):
    g = g.copy()
    g["instrument"] = instrument  # keep instrument explicitly as a column
    
    labeled_g = apply_triple_barrier_to_instrument(
        g,
        price_col=price_col,
        vol_col=vol_col,
        horizon=HORIZON,
        pt_mult=PT_MULT,
        sl_mult=SL_MULT
    )
    
    labeled_g["instrument"] = instrument
    labeled_groups.append(labeled_g)

meta_labeled_df = (
    pd.concat(labeled_groups, axis=0)
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Final labeled dataset shape:")
print(meta_labeled_df.shape)

print("\nColumns check:")
print("instrument" in meta_labeled_df.columns)

print("\nMeta-label distribution:")
display(
    meta_labeled_df["meta_label"]
    .value_counts(dropna=False)
)

print("\nBarrier event distribution:")
display(
    meta_labeled_df["tb_event_type"]
    .value_counts(dropna=False)
)

Final labeled dataset shape:
(1237, 155)

Columns check:
True

Meta-label distribution:


meta_label
1.0    690
0.0    543
NaN      4
Name: count, dtype: int64


Barrier event distribution:


tb_event_type
profit_taking       625
stop_loss           490
vertical_barrier    118
no_future_data        4
Name: count, dtype: int64

In [90]:
display(
    meta_df[
        meta_labeled_df["tb_event_type"]=="no_future_data"
    ][
        [
            "instrument",
            "date",
            "primary_signal"
        ]
    ]
)

,instrument,date,primary_signal
421,cl1s,2022-06-30,1
484,ho1s,2022-06-02,-1
608,ng1s,2022-06-29,-1
1236,rb1s,2022-06-30,1


## 3.7 Remove Unlabelable Observations

A very small number of observations occur at the end of the available sample and do not contain enough future information to reach the full vertical barrier horizon.

These observations cannot be labeled consistently and are therefore removed.

This does not introduce information leakage because the future path required for label construction simply does not exist.

In [91]:
# ============================================================
# Phase 3.7 — Remove Unlabelable Observations
# ============================================================

n_before = len(meta_labeled_df)

meta_labeled_df = (
    meta_labeled_df
    .dropna(subset=["meta_label"])
    .reset_index(drop=True)
)

n_after = len(meta_labeled_df)

print("Rows before cleaning:", n_before)
print("Rows after cleaning:", n_after)

print(
    f"\nRemoved observations: {n_before-n_after}"
)

print(
    f"Percentage removed: "
    f"{100*(n_before-n_after)/n_before:.3f}%"
)

Rows before cleaning: 1237
Rows after cleaning: 1233

Removed observations: 4
Percentage removed: 0.323%


In [92]:
# ============================================================
# Phase 3.8 — Label Distribution by Instrument
# ============================================================

print("Meta-label distribution by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["meta_label"],
        margins=True
    )
)

print("\nPercentages by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["meta_label"],
        normalize="index"
    ).round(3)
)

Meta-label distribution by instrument:


meta_label,0.0,1.0,All
instrument,,,
cl1s,150,271,421
ho1s,26,36,62
ng1s,66,57,123
rb1s,301,326,627
All,543,690,1233



Percentages by instrument:


meta_label,0.0,1.0
instrument,,
cl1s,0.356,0.644
ho1s,0.419,0.581
ng1s,0.537,0.463
rb1s,0.480,0.520


In [93]:
# ============================================================
# Phase 3.9 — Barrier Event Distribution by Instrument
# ============================================================

print("Barrier event distribution by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["tb_event_type"],
        margins=True
    )
)

print("\nPercentages by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["tb_event_type"],
        normalize="index"
    ).round(3)
)

Barrier event distribution by instrument:


tb_event_type,profit_taking,stop_loss,vertical_barrier,All
instrument,,,,
cl1s,247,138,36,421
ho1s,36,24,2,62
ng1s,48,61,14,123
rb1s,294,267,66,627
All,625,490,118,1233



Percentages by instrument:


tb_event_type,profit_taking,stop_loss,vertical_barrier
instrument,,,
cl1s,0.587,0.328,0.086
ho1s,0.581,0.387,0.032
ng1s,0.390,0.496,0.114
rb1s,0.469,0.426,0.105


# Phase 4 — Meta-Model Development

We now move from label construction to supervised learning.

The objective is to train a global meta-model across all instruments. Each observation corresponds to an active primary signal, and the target is the triple-barrier meta-label:

$$
\text{meta label}=1
$$

if the primary trade was profitable under the triple-barrier rule, and:

$$
\text{meta label}=0
$$

otherwise.

The model will use market features, the primary signal direction, and the instrument identity to learn when the primary model should be trusted.

Importantly, columns generated by the triple-barrier procedure are not used as predictors, since they are constructed using future price information.

## 4.1 Defining Target and Remove Leakage Columns

In [94]:
# ============================================================
# Phase 4.1 — Define Target and Remove Leakage Columns
# ============================================================

ml_df = meta_labeled_df.copy()

target_col = "meta_label"

# Columns generated by the labeling procedure.
# These must NOT be used as ML features because they use future information.
leakage_cols = [
    "meta_label",
    "tb_event_type",
    "tb_exit_date",
    "tb_exit_price",
    "tb_trade_return"
]

# Date is not used directly as a feature.
# It is kept in ml_df only for chronological splitting and CV construction.
non_feature_cols = [
    "hmm_prob_sum"
]

excluded_cols = leakage_cols + non_feature_cols

feature_cols = [
    col for col in ml_df.columns
    if col not in excluded_cols
]

# Replace infinite feature values by NaN directly in ml_df.
# The imputers inside the preprocessing pipelines will handle them later.
ml_df[feature_cols] = ml_df[feature_cols].replace([np.inf, -np.inf], np.nan)

X = ml_df[feature_cols].copy()
y = ml_df[target_col].astype(int).copy()

print("ML dataframe shape:", ml_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
display(y.value_counts(normalize=False))

print("\nTarget distribution (%):")
display((100 * y.value_counts(normalize=True)).round(2))

print("\nFeature columns:")
display(feature_cols)

ML dataframe shape: (1233, 155)
Feature matrix shape: (1233, 149)
Target shape: (1233,)

Target distribution:


meta_label
1    690
0    543
Name: count, dtype: int64


Target distribution (%):


meta_label
1    55.96
0    44.04
Name: proportion, dtype: float64


Feature columns:


['date',
 'instrument',
 'primary_signal',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'open_interest',
 'log_return',
 'momentum_5d',
 'momentum_10d',
 'momentum_20d',
 'momentum_60d',
 'mean_return_5d',
 'mean_return_20d',
 'mean_return_60d',
 'ret_20d_zscore',
 'realized_vol_5d',
 'realized_vol_20d',
 'realized_vol_60d',
 'ewma_vol_10d',
 'ewma_vol_20d',
 'vol_ratio_20_60',
 'garman_klass_vol',
 'yang_zhang_vol',
 'skew_20d',
 'skew_60d',
 'kurt_20d',
 'downside_vol_20d',
 'price_range_position_60d',
 'range_position_5d_chg',
 'return_vol_correl_60d',
 'vol_parkinson_20d',
 'vol_rogers_satchell_20d',
 'log_volume_change',
 'volume_mean_5d',
 'volume_std_5d',
 'volume_mean_20d',
 'volume_std_20d',
 'volume_zscore_20d',
 'log_oi_change',
 'oi_momentum_5d',
 'oi_momentum_20d',
 'amihud_20d',
 'roll_spread_20d',
 'dollar_volume_log',
 'kyle_lambda_20d',
 'hl_spread',
 'oc_spread',
 'close_position',
 'rsi_14d',
 'price_zscore_20d',
 'price_zscore_60d',
 'macd',
 'macd_signal',
 'mac

## 4.2 Missing Values Check

Before constructing the machine learning pipelines, we verify whether any missing values remain in the feature matrix.

Several features rely on rolling windows, autocorrelation estimates, and regime detection models. Missing values may therefore remain after the feature engineering phase.

We inspect the remaining missing values before deciding whether to drop observations or apply imputation.

In [95]:
# ============================================================
# Phase 4.2 — Missing Values Check
# ============================================================

missing_counts = X.isna().sum()

missing_counts = (
    missing_counts[missing_counts > 0]
    .sort_values(ascending=False)
)

print("Number of features with missing values:")
print(len(missing_counts))

if len(missing_counts) > 0:
    display(missing_counts)

    print(
        "\nPercentage missing:"
    )

    display(
        (
            100 * missing_counts / len(X)
        ).round(3)
    )
else:
    print("No missing values found.")

Number of features with missing values:
16


multiasset_vol_zscore_252d    1233
relative_energy_vol            639
leadlag_anchor                 421
asset_equity_corr_60d          102
signal_trend_concord            82
multiasset_vol_index            60
equity_vol_avg_20d              60
signal_density_20d              29
amihud_20d                      17
kyle_lambda_20d                 17
oi_momentum_5d                   2
oi_momentum_20d                  2
log_volume_change                1
log_oi_change                    1
dollar_volume_log                1
vol_oi_ratio                     1
dtype: int64


Percentage missing:


multiasset_vol_zscore_252d    100.000
relative_energy_vol            51.825
leadlag_anchor                 34.144
asset_equity_corr_60d           8.273
signal_trend_concord            6.650
multiasset_vol_index            4.866
equity_vol_avg_20d              4.866
signal_density_20d              2.352
amihud_20d                      1.379
kyle_lambda_20d                 1.379
oi_momentum_5d                  0.162
oi_momentum_20d                 0.162
log_volume_change               0.081
log_oi_change                   0.081
dollar_volume_log               0.081
vol_oi_ratio                    0.081
dtype: float64

If there is a feature with more than 1% missing values, we will drop the feature.

In [96]:
high_missing = missing_counts[missing_counts > 0.01 * len(X)].index.tolist()
print("Features with more than 1% missing values:")
display(high_missing)
X = X.drop(columns=high_missing)

Features with more than 1% missing values:


['multiasset_vol_zscore_252d',
 'relative_energy_vol',
 'leadlag_anchor',
 'asset_equity_corr_60d',
 'signal_trend_concord',
 'multiasset_vol_index',
 'equity_vol_avg_20d',
 'signal_density_20d',
 'amihud_20d',
 'kyle_lambda_20d']

In [97]:
print("Number of infinite values remaining:")
print(np.isinf(X.select_dtypes(include=np.number)).sum().sum())

Number of infinite values remaining:
0


## 4.3 Split into train and test

We are not going to recompute the split because we are going to use the global split used before.

In [98]:
train_mask = ml_df["date"] <= global_train_end_date
test_mask  = ml_df["date"] >  global_train_end_date

X_train = X.loc[train_mask]
X_test  = X.loc[test_mask]
y_train = y.loc[train_mask].values.astype(int) # Check later :))))))))
y_test  = y.loc[test_mask].values.astype(int)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Features names: {X_train.columns}")
print(f"Distribution y_train : {np.bincount(y_train)}")
print(f"Distribution y_test  : {np.bincount(y_test)}")
print(f"Shape y_train : {y_train.shape}")
print(f"Shape y_test  : {y_test.shape}")

X_train : (965, 139)
X_test  : (268, 139)
Features names: Index(['date', 'instrument', 'primary_signal', 'open', 'high', 'low', 'close', 'volume', 'open_interest', 'log_return',
       ...
       'hmm_prob_state_1', 'hmm_prob_state_2', 'hg_ret_daily', 'equity_ret_daily', 'hmm_macro_prob_state_0',
       'hmm_macro_prob_state_1', 'hmm_macro_regime', 'hmm_macro_prob_sum', 'signal_changed', 'signal_persistence'],
      dtype='object', length=139)
Distribution y_train : [433 532]
Distribution y_test  : [110 158]
Shape y_train : (965,)
Shape y_test  : (268,)
